# NB04: Putirka benchmark with dual tables (fair + practical)

## Purpose / Inputs / Outputs / Canonical decisions

**Purpose:** Benchmark the canonical opx-liq ML model against the Putirka (2008) thermobarometer using `Thermobar`. Reports a unified table with three scopes (full / Putirka-valid subset / intersection), paired Wilcoxon tests on |residuals|, Delta-RMSE bootstrap CIs, and a failure-mode Mann-Whitney on Putirka-fail vs Putirka-valid.

**Inputs:** `data/processed/opx_clean_opx_liq.parquet`, `results/nb03_winning_configurations.json`, canonical RF models.

**Outputs:** `results/nb04_putirka_comparison.csv` (+ `_fair`, `_practical`, `_unified`), `results/nb04_putirka_vs_ml.csv` (legacy schema for NBF fig10), `results/nb04_putirka_vs_ml_wilcoxon.csv`, `figures/fig_nb04_putirka_comparison.{png,pdf}`.

**Canonical decisions:** Uses Putirka Option 2 (passing true observed P and T as inputs) as the canonical Putirka scope for the legacy comparison CSV. The OPERATOR DECISION block (A/B/C) controls which scope becomes the manuscript headline.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (
    ROOT, DATA_RAW, DATA_EXTERNAL, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
    MODELS, FIGURES, RESULTS, LOGS,
    EXPETDB, LEPR_XLSX, LIN2023_NATURAL,
    FE3_FET_RATIO, KD_FEMG_MIN, KD_FEMG_MAX, WO_MAX_MOL_PCT,
    P_CEILING_KBAR, CATION_SUM_MIN, CATION_SUM_MAX,
    OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX,
    SEED_SPLIT, SEED_MODEL, SEED_NOISE_AUG, SEED_KMEANS,
    OPX_RAW_OXIDES, OPX_FULL_OXIDES, LIQ_OXIDES,
    SEED_BOOTSTRAP,
)
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import Thermobar as pt

In [ ]:
# Canonical features and prediction helpers from src/ (one source of truth).
from src.features import (
    build_feature_matrix,
    make_raw_features,
    make_alr_features,
    make_pwlr_features,
    augment_dataframe,
)
from src.models import predict_median, predict_iqr


In [ ]:
# v7 Part H: forest-family canonical load for fair comparison
# Earlier versions selected the RF model per feature set directly from the
# multi-seed summary. v7 consolidates that choice into the per-family
# winner JSON written by nb03 Phase 3.5; we now load the canonical forest
# joblib (RF or ERT depending on the tiebreaker outcome).
from src.data import canonical_model_path, canonical_model_spec

spec_T = canonical_model_spec('T_C', 'opx_liq', 'forest', RESULTS)
spec_P = canonical_model_spec('P_kbar', 'opx_liq', 'forest', RESULTS)
fs_T = spec_T['feature_set']
fs_P = spec_P['feature_set']
print(f"Forest-family opx-liq: T uses {spec_T['model_name']}/{fs_T}, "
      f"P uses {spec_P['model_name']}/{fs_P}")

model_T_ml = joblib.load(canonical_model_path('T_C', 'opx_liq', 'forest',
                                              MODELS, RESULTS))
model_P_ml = joblib.load(canonical_model_path('P_kbar', 'opx_liq', 'forest',
                                              MODELS, RESULTS))


In [ ]:
# Load opx-liq test set
df_liq = pd.read_parquet(DATA_PROC / 'opx_clean_opx_liq.parquet')
test_idx = np.load(DATA_SPLITS / 'test_indices_opx_liq.npy')
df_test = df_liq.loc[test_idx].copy()
print(f'Test set: {len(df_test)} experiments from {df_test["Citation"].nunique()} studies')

In [ ]:
# Format for Thermobar
fe3_array = np.full(len(df_test), FE3_FET_RATIO)

if 'H2O_Liq' in df_test.columns:
    h2o_array = df_test['H2O_Liq'].fillna(0).values.astype(float)
else:
    h2o_array = np.zeros(len(df_test))
    print('WARNING: H2O_Liq not found, using zeros')

def g(col, default=0):
    if col in df_test.columns:
        return df_test[col].fillna(default).values.astype(float)
    return np.full(len(df_test), default, dtype=float)

opx_test = pd.DataFrame({
    'SiO2_Opx': g('SiO2'),
    'TiO2_Opx': g('TiO2'),
    'Al2O3_Opx': g('Al2O3'),
    'FeOt_Opx': g('FeO_total'),
    'MgO_Opx': g('MgO'),
    'CaO_Opx': g('CaO'),
    'MnO_Opx': g('MnO'),
    'Cr2O3_Opx': g('Cr2O3'),
    'Na2O_Opx': g('Na2O'),
})

liq_test = pd.DataFrame({
    'SiO2_Liq': g('liq_SiO2'),
    'TiO2_Liq': g('liq_TiO2'),
    'Al2O3_Liq': g('liq_Al2O3'),
    'FeOt_Liq': g('liq_FeO'),
    'MgO_Liq': g('liq_MgO'),
    'CaO_Liq': g('liq_CaO'),
    'Na2O_Liq': g('liq_Na2O'),
    'K2O_Liq': g('liq_K2O'),
    'MnO_Liq': g('liq_MnO' if 'liq_MnO' in df_test.columns else 'FeO_total') * 0,
    'Cr2O3_Liq': np.zeros(len(df_test)),
    'P2O5_Liq': np.zeros(len(df_test)),
    'Fe3Fet_Liq': fe3_array,
    'H2O_Liq': h2o_array,
})

y_T_true = df_test['T_C'].values.astype(float)
y_P_true = df_test['P_kbar'].values.astype(float)

In [ ]:
# Helper to coerce Thermobar output to numpy array
def to_array(x, key_T=False):
    if isinstance(x, pd.DataFrame):
        # Try common column names
        for col in ['T_K_calc', 'T_C_calc', 'T_K', 'T_C', 'Temperature_K',
                    'P_kbar_calc', 'P_kbar', 'Pressure_kbar']:
            if col in x.columns:
                arr = x[col].values.astype(float)
                if 'T_K' in col or col == 'Temperature_K':
                    arr = arr - 273.15
                return arr
        return x.iloc[:, 0].values.astype(float)
    if isinstance(x, pd.Series):
        arr = x.values.astype(float)
    else:
        arr = np.asarray(x, dtype=float)
    return arr

def safe_thermobar(func, **kwargs):
    try:
        return to_array(func(**kwargs))
    except Exception as e:
        print(f'  {func.__name__} FAILED: {e}')
        return np.full(len(df_test), np.nan)

In [ ]:
# Option 2: true P/T as input
print('Option 2 (true P/T as input):')
T_put_28a = safe_thermobar(
    pt.calculate_opx_liq_temp,
    equationT='T_Put2008_eq28a', opx_comps=opx_test, liq_comps=liq_test,
    P=y_P_true, Fe3Fet_Liq=fe3_array, H2O_Liq=h2o_array,
)
T_put_28b = safe_thermobar(
    pt.calculate_opx_liq_temp,
    equationT='T_Put2008_eq28b_opx_sat', opx_comps=opx_test, liq_comps=liq_test,
    P=y_P_true, Fe3Fet_Liq=fe3_array, H2O_Liq=h2o_array,
)
P_put_29a = safe_thermobar(
    pt.calculate_opx_liq_press,
    equationP='P_Put2008_eq29a', opx_comps=opx_test, liq_comps=liq_test,
    T=y_T_true + 273.15, Fe3Fet_Liq=fe3_array, H2O_Liq=h2o_array,
)
P_put_29c = safe_thermobar(
    pt.calculate_opx_only_press,
    equationP='P_Put2008_eq29c', opx_comps=opx_test, T=y_T_true + 273.15,
)
# Thermobar T_Put2008_eq28a/28b return T in Kelvin; convert to Celsius
T_put_28a = T_put_28a - 273.15
T_put_28b = T_put_28b - 273.15

# Sanity bounds: mark physically implausible Putirka outputs as failures.
# Putirka 28a/29a are unstable for some opx-liq pairs and produce extreme outliers.
# Reasonable bounds: T in 400-1900 C, P in -2 to 100 kbar (matches training range).
def clip_to_failure(arr, lo, hi):
    arr = np.asarray(arr, dtype=float)
    bad = (arr < lo) | (arr > hi)
    arr[bad] = np.nan
    return arr

T_put_28a = clip_to_failure(T_put_28a, 400, 1900)
T_put_28b = clip_to_failure(T_put_28b, 400, 1900)
P_put_29a = clip_to_failure(P_put_29a, -2, 100)
P_put_29c = clip_to_failure(P_put_29c, -2, 100)

print(f'  28a: n_valid={np.isfinite(T_put_28a).sum()}, mean={np.nanmean(T_put_28a):.0f}')
print(f'  28b: n_valid={np.isfinite(T_put_28b).sum()}, mean={np.nanmean(T_put_28b):.0f}')
print(f'  29a: n_valid={np.isfinite(P_put_29a).sum()}, mean={np.nanmean(P_put_29a):.1f}')
print(f'  29c: n_valid={np.isfinite(P_put_29c).sum()}, mean={np.nanmean(P_put_29c):.1f}')

In [ ]:
# Option 1: iterative solver
print('Option 1 (iterative):')
try:
    iter_result = pt.calculate_opx_liq_press_temp(
        equationT='T_Put2008_eq28a',
        equationP='P_Put2008_eq29a',
        opx_comps=opx_test, liq_comps=liq_test,
        Fe3Fet_Liq=fe3_array, H2O_Liq=h2o_array,
        iterations=30,
    )
    if isinstance(iter_result, pd.DataFrame):
        print('  columns:', iter_result.columns.tolist()[:10])
        T_col = next((c for c in ['T_K_calc','T_C_calc','T_K','Temperature_K'] if c in iter_result.columns), None)
        P_col = next((c for c in ['P_kbar_calc','P_kbar','Pressure_kbar'] if c in iter_result.columns), None)
        T_put_iter = iter_result[T_col].values.astype(float) if T_col else np.full(len(df_test), np.nan)
        P_put_iter = iter_result[P_col].values.astype(float) if P_col else np.full(len(df_test), np.nan)
        if T_col and 'K' in T_col:
            T_put_iter = T_put_iter - 273.15
    else:
        T_put_iter = np.full(len(df_test), np.nan)
        P_put_iter = np.full(len(df_test), np.nan)
    T_put_iter = clip_to_failure(T_put_iter, 400, 1900)
    P_put_iter = clip_to_failure(P_put_iter, -2, 100)
    print(f'  T_iter n_valid={np.isfinite(T_put_iter).sum()}, mean={np.nanmean(T_put_iter):.0f}')
    print(f'  P_iter n_valid={np.isfinite(P_put_iter).sum()}, mean={np.nanmean(P_put_iter):.1f}')
except Exception as e:
    print(f'  iterative FAILED: {e}')
    T_put_iter = np.full(len(df_test), np.nan)
    P_put_iter = np.full(len(df_test), np.nan)

In [ ]:
# ML predictions + Option 3 (ML P/T fed to Putirka)
X_test_T, _ = build_feature_matrix(df_test, fs_T, use_liq=True)
X_test_P, _ = build_feature_matrix(df_test, fs_P, use_liq=True)
T_ml_pred = model_T_ml.predict(X_test_T)
P_ml_pred = model_P_ml.predict(X_test_P)

print(f'ML: T RMSE={np.sqrt(mean_squared_error(y_T_true, T_ml_pred)):.2f}, R2={r2_score(y_T_true, T_ml_pred):.3f}')
print(f'ML: P RMSE={np.sqrt(mean_squared_error(y_P_true, P_ml_pred)):.2f}, R2={r2_score(y_P_true, P_ml_pred):.3f}')

print('Option 3 (ML-P fed to Putirka T, ML-T fed to Putirka P):')
T_put_mlP = safe_thermobar(
    pt.calculate_opx_liq_temp,
    equationT='T_Put2008_eq28a', opx_comps=opx_test, liq_comps=liq_test,
    P=P_ml_pred, Fe3Fet_Liq=fe3_array, H2O_Liq=h2o_array,
)
P_put_mlT = safe_thermobar(
    pt.calculate_opx_liq_press,
    equationP='P_Put2008_eq29a', opx_comps=opx_test, liq_comps=liq_test,
    T=T_ml_pred + 273.15, Fe3Fet_Liq=fe3_array, H2O_Liq=h2o_array,
)
T_put_mlP = T_put_mlP - 273.15
T_put_mlP = clip_to_failure(T_put_mlP, 400, 1900)
P_put_mlT = clip_to_failure(P_put_mlT, -2, 100)
print(f'  T_put_mlP mean={np.nanmean(T_put_mlP):.0f}')
print(f'  P_put_mlT mean={np.nanmean(P_put_mlT):.1f}')

In [ ]:
# Dual tables: Fair and Practical
def compute_row(method, target, y_true, y_pred, valid_mask):
    yt = y_true[valid_mask]; yp = y_pred[valid_mask]
    finite = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[finite], yp[finite]
    if len(yt) == 0:
        return {'Method': method, 'Target': target, 'RMSE': np.nan,
                'MAE': np.nan, 'R2': np.nan, 'n_test': 0}
    return {
        'Method': method, 'Target': target,
        'RMSE': np.sqrt(mean_squared_error(yt, yp)),
        'MAE': mean_absolute_error(yt, yp),
        'R2': r2_score(yt, yp),
        'n_test': len(yt),
    }

T_arrays = {
    'Putirka 28a (true P)': T_put_28a,
    'Putirka 28b_opx_sat (true P)': T_put_28b,
    'Putirka iterative (28a+29a)': T_put_iter,
    'Putirka 28a (ML P)': T_put_mlP,
    'ML-RF opx-liq': T_ml_pred,
}
P_arrays = {
    'Putirka 29a (true T)': P_put_29a,
    'Putirka 29c (opx-only, true T)': P_put_29c,
    'Putirka iterative (28a+29a)': P_put_iter,
    'Putirka 29a (ML T)': P_put_mlT,
    'ML-RF opx-liq': P_ml_pred,
}

T_masks = [np.isfinite(arr) for arr in T_arrays.values()]
P_masks = [np.isfinite(arr) for arr in P_arrays.values()]
T_all_valid = np.logical_and.reduce(T_masks)
P_all_valid = np.logical_and.reduce(P_masks)
print(f'T: all-method valid on {T_all_valid.sum()}/{len(y_T_true)}')
print(f'P: all-method valid on {P_all_valid.sum()}/{len(y_P_true)}')

# Fair table: same sample subset for all methods
fair_rows = []
for name, arr in T_arrays.items():
    fair_rows.append(compute_row(name, 'T', y_T_true, arr, T_all_valid))
for name, arr in P_arrays.items():
    fair_rows.append(compute_row(name, 'P', y_P_true, arr, P_all_valid))
fair_df = pd.DataFrame(fair_rows)
fair_df.to_csv(RESULTS / 'nb04_putirka_comparison_fair.csv', index=False)
fair_df.to_csv(RESULTS / 'nb04_putirka_comparison.csv', index=False)
print('\nFAIR comparison:')
print(fair_df.round(3).to_string(index=False))

In [ ]:
# Practical table: ML on all samples, Putirka failure rate
practical_rows = []
practical_rows.append({
    'Method': 'ML-RF opx-liq (all test)', 'Target': 'T',
    'RMSE': np.sqrt(mean_squared_error(y_T_true, T_ml_pred)),
    'MAE': mean_absolute_error(y_T_true, T_ml_pred),
    'R2': r2_score(y_T_true, T_ml_pred),
    'n_test': len(y_T_true),
    'failure_pct': 0.0,
})
practical_rows.append({
    'Method': 'ML-RF opx-liq (all test)', 'Target': 'P',
    'RMSE': np.sqrt(mean_squared_error(y_P_true, P_ml_pred)),
    'MAE': mean_absolute_error(y_P_true, P_ml_pred),
    'R2': r2_score(y_P_true, P_ml_pred),
    'n_test': len(y_P_true),
    'failure_pct': 0.0,
})
practical_rows.append({
    'Method': 'Putirka failure rate', 'Target': 'T',
    'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan,
    'n_test': int((~T_all_valid).sum()),
    'failure_pct': 100.0 * (~T_all_valid).sum() / len(y_T_true),
})
practical_rows.append({
    'Method': 'Putirka failure rate', 'Target': 'P',
    'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan,
    'n_test': int((~P_all_valid).sum()),
    'failure_pct': 100.0 * (~P_all_valid).sum() / len(y_P_true),
})
practical_df = pd.DataFrame(practical_rows)
practical_df.to_csv(RESULTS / 'nb04_putirka_comparison_practical.csv', index=False)
print('PRACTICAL comparison:')
print(practical_df.round(3).to_string(index=False))

In [ ]:
# Legacy CSV for NBF fig 10 (schema: obs_*, putirka_pred_*, ml_pred_*).
# Option 2 Putirka predictions (true P/T as input) are the canonical Putirka
# column since that is the configuration reported as the Putirka benchmark.
legacy_df = pd.DataFrame({
    'obs_T_C':             y_T_true,
    'obs_P_kbar':          y_P_true,
    'putirka_pred_T_C':    T_put_28a,
    'putirka_pred_P_kbar': P_put_29a,
    'ml_pred_T_C':         T_ml_pred,
    'ml_pred_P_kbar':      P_ml_pred,
})
legacy_df.to_csv(RESULTS / 'nb04_putirka_vs_ml.csv', index=False)
print(f'Wrote legacy {RESULTS.name}/nb04_putirka_vs_ml.csv ({len(legacy_df)} rows)')


<!-- v7-fix-section-header -->
## Part 1: ExPetDB internal test benchmark

**Purpose.** Compare the canonical opx-liq ML against Putirka 2008 eq28a/29a on
the internal ExPetDB held-out test set. Intersection-scope comparison (same
rows) isolates model quality from coverage.

**Data inputs.** `data/processed/opx_clean_opx_liq.parquet` test rows selected
by `data/splits/test_indices_opx_liq.npy` (n~=174). `results/nb03_per_family_winners.json`
for the canonical model/feature choice.

**Methods evaluated.**
1. Putirka 2008 opx-liq, Option 1 iterative (eq 28a + eq 29a)
2. Putirka 2008 opx-liq, Option 2 true P/T as input (ceiling)
3. Putirka 2008 opx-liq, Option 3 ML P/T fed back to Putirka
4. Ours canonical opx-liq forest (RF on alr for T, RF on raw for P)
5. Ours canonical opx-liq boosted (XGB on raw for both)

**Analysis performed.** Intersection-scope RMSE + 95% bootstrap CI (B=2000),
failure-rate analysis for Putirka, paired Wilcoxon signed-rank of ML vs each
Putirka variant, compositional stratification of failure modes.

**How to interpret.** Intersection RMSE compares methods on equal footing.
A substantially larger Putirka failure rate than ML indicates deployment
coverage matters even before looking at RMSE.

**Outputs.**
- `results/nb04_unified_benchmark.csv`
- `results/nb04_paired_wilcoxon.csv`
- `results/nb04_failure_mode_stratification.csv`
- `figures/fig_nb04_failure_analysis.{png,pdf}`
- `figures/fig_nb04_diagnostic_encyclopedia_T.{png,pdf}`, `fig_nb04_diagnostic_encyclopedia_P.{png,pdf}`

**Downstream use.** NB09 manuscript compilation reads the unified table.


In [ ]:
# Part 1a: unified_benchmark_table (three denominators)
from scipy import stats

BOOT_B = 2000
BOOT_RNG = np.random.default_rng(SEED_BOOTSTRAP)

# Putirka arrays already defined above. ML arrays: T_ml_pred, P_ml_pred.
T_methods = {
    'Putirka 28a (true P)':         T_put_28a,
    'Putirka 28b_opx_sat (true P)': T_put_28b,
    'Putirka iterative (28a+29a)':  T_put_iter,
    'Putirka 28a (ML P)':           T_put_mlP,
    'ML-RF opx-liq':                T_ml_pred,
}
P_methods = {
    'Putirka 29a (true T)':           P_put_29a,
    'Putirka 29c (opx-only, true T)': P_put_29c,
    'Putirka iterative (28a+29a)':    P_put_iter,
    'Putirka 29a (ML T)':             P_put_mlT,
    'ML-RF opx-liq':                  P_ml_pred,
}

# Define scopes.
T_putirka_valid = (np.isfinite(T_put_28a) & np.isfinite(T_put_28b) &
                   np.isfinite(T_put_iter) & np.isfinite(T_put_mlP))
P_putirka_valid = (np.isfinite(P_put_29a) & np.isfinite(P_put_29c) &
                   np.isfinite(P_put_iter) & np.isfinite(P_put_mlT))

T_intersection = T_putirka_valid & np.isfinite(T_ml_pred) & np.isfinite(y_T_true)
P_intersection = P_putirka_valid & np.isfinite(P_ml_pred) & np.isfinite(y_P_true)

N_full = len(y_T_true)


def rmse_with_ci(y_true, y_pred, mask, B=BOOT_B, rng=BOOT_RNG):
    m = mask & np.isfinite(y_true) & np.isfinite(y_pred)
    if m.sum() < 2:
        return np.nan, np.nan, np.nan, int(m.sum())
    yt = y_true[m]; yp = y_pred[m]
    n = len(yt)
    point = float(np.sqrt(np.mean((yt - yp) ** 2)))
    idx = rng.integers(0, n, size=(B, n))
    boots = np.sqrt(np.mean((yt[idx] - yp[idx]) ** 2, axis=1))
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return point, float(lo), float(hi), n


unified_rows = []
for target, methods, y_true, scopes in [
    ('T', T_methods, y_T_true,
     {'full':           np.isfinite(y_T_true),
      'putirka_valid':  T_putirka_valid,
      'intersection':   T_intersection}),
    ('P', P_methods, y_P_true,
     {'full':           np.isfinite(y_P_true),
      'putirka_valid':  P_putirka_valid,
      'intersection':   P_intersection}),
]:
    for scope_name, scope_mask in scopes.items():
        for method_name, pred in methods.items():
            rmse, lo, hi, n = rmse_with_ci(y_true, pred, scope_mask)
            unified_rows.append({
                'target':        target,
                'scope':         scope_name,
                'method':        method_name,
                'n':             n,
                'rmse':          rmse,
                'rmse_ci_lo':    lo,
                'rmse_ci_hi':    hi,
                'failure_pct':   (100.0 * (~np.isfinite(pred)).sum() / N_full
                                  if method_name.startswith('Putirka') else 0.0),
            })

unified_df = pd.DataFrame(unified_rows)
unified_df.to_csv(RESULTS / 'nb04_unified_benchmark.csv', index=False)

print(f'N_full = {N_full}')
print(f'T putirka_valid = {int(T_putirka_valid.sum())}, '
      f'P putirka_valid = {int(P_putirka_valid.sum())}')
print(f'T intersection  = {int(T_intersection.sum())}, '
      f'P intersection  = {int(P_intersection.sum())}')
print('\nUnified benchmark (RMSE with 95% bootstrap CI, B=2000):')
for tgt in ['T', 'P']:
    print(f'\n--- target = {tgt} ---')
    sub = unified_df[unified_df.target == tgt].copy()
    sub['rmse_str'] = sub.apply(
        lambda r: (f"{r['rmse']:.2f} [{r['rmse_ci_lo']:.2f}, {r['rmse_ci_hi']:.2f}]"
                   if np.isfinite(r['rmse']) else 'n/a'), axis=1)
    print(sub[['scope', 'method', 'n', 'rmse_str', 'failure_pct']]
          .to_string(index=False))


In [ ]:
# Part 1b: paired_wilcoxon (ML vs each Putirka variant, intersection scope,
# plus a paired bootstrap CI on the Delta-RMSE).
wilcoxon_rows = []

for target, methods, y_true, inter_mask in [
    ('T', T_methods, y_T_true, T_intersection),
    ('P', P_methods, y_P_true, P_intersection),
]:
    ml_pred = methods['ML-RF opx-liq']
    yt_i  = y_true[inter_mask]
    ml_i  = ml_pred[inter_mask]
    ml_abs_res = np.abs(yt_i - ml_i)
    for method_name, pred in methods.items():
        if method_name == 'ML-RF opx-liq':
            continue
        pu_i = pred[inter_mask]
        mask_ok = np.isfinite(ml_i) & np.isfinite(pu_i) & np.isfinite(yt_i)
        if mask_ok.sum() < 3:
            wilcoxon_rows.append({
                'target': target, 'ml_vs': method_name,
                'n_pairs': int(mask_ok.sum()),
                'wilcoxon_stat': np.nan, 'p_value': np.nan,
                'delta_rmse': np.nan, 'delta_ci_lo': np.nan, 'delta_ci_hi': np.nan,
            })
            continue
        yt = yt_i[mask_ok]
        ml_r = np.abs(yt - ml_i[mask_ok])
        pu_r = np.abs(yt - pu_i[mask_ok])
        try:
            stat, pval = stats.wilcoxon(ml_r, pu_r, zero_method='wilcox',
                                        alternative='two-sided')
        except ValueError:
            stat, pval = np.nan, np.nan
        # Paired bootstrap on Delta-RMSE = RMSE(pu) - RMSE(ml), so positive means
        # ML is better.
        n = len(yt)
        idx = BOOT_RNG.integers(0, n, size=(BOOT_B, n))
        yb = yt[idx]
        mb = ml_i[mask_ok][idx]
        pb = pu_i[mask_ok][idx]
        delta = (np.sqrt(np.mean((yb - pb) ** 2, axis=1)) -
                 np.sqrt(np.mean((yb - mb) ** 2, axis=1)))
        d_lo, d_hi = np.quantile(delta, [0.025, 0.975])
        delta_point = (float(np.sqrt(np.mean((yt - pu_i[mask_ok]) ** 2))) -
                       float(np.sqrt(np.mean((yt - ml_i[mask_ok]) ** 2))))
        wilcoxon_rows.append({
            'target':       target,
            'ml_vs':        method_name,
            'n_pairs':      int(mask_ok.sum()),
            'wilcoxon_stat': float(stat) if np.isfinite(stat) else np.nan,
            'p_value':      float(pval) if np.isfinite(pval) else np.nan,
            'delta_rmse':   float(delta_point),
            'delta_ci_lo':  float(d_lo),
            'delta_ci_hi':  float(d_hi),
        })

wilcoxon_df = pd.DataFrame(wilcoxon_rows)
wilcoxon_df.to_csv(RESULTS / 'nb04_paired_wilcoxon.csv', index=False)
print('Paired Wilcoxon signed-rank on |residuals| (ML vs Putirka variant,')
print(f'intersection scope, B={BOOT_B} paired bootstrap for Delta-RMSE CI):')
print('delta_rmse > 0 => ML is better (lower error).\n')
print(wilcoxon_df.round(4).to_string(index=False))


In [ ]:
# Part 1c: failure_mode_strat (what composition features separate the samples
# where Putirka fails from the ones where it converges?)
fail_features = ['SiO2', 'Al2O3', 'MgO', 'CaO', 'FeO_total',
                 'liq_SiO2', 'liq_MgO', 'liq_Al2O3', 'liq_CaO',
                 'T_C', 'P_kbar']
fail_features = [c for c in fail_features if c in df_test.columns]

# A sample is "Putirka-fails" if ANY Putirka output (28a OR 29a) is NaN.
putirka_fails = ~(np.isfinite(T_put_28a) & np.isfinite(P_put_29a))
stratify_df = df_test[fail_features].copy()
stratify_df['putirka_fails'] = putirka_fails

strat_rows = []
for feat in fail_features:
    vals = stratify_df[feat].astype(float).values
    ok = np.isfinite(vals)
    conv = vals[ok & ~putirka_fails]
    fail = vals[ok &  putirka_fails]
    if len(conv) < 3 or len(fail) < 3:
        strat_rows.append({'feature': feat, 'n_converged': len(conv),
                           'n_failed': len(fail),
                           'mean_converged': np.nan, 'mean_failed': np.nan,
                           'mannwhitney_U': np.nan, 'p_value': np.nan})
        continue
    U, pval = stats.mannwhitneyu(conv, fail, alternative='two-sided')
    strat_rows.append({
        'feature':        feat,
        'n_converged':    int(len(conv)),
        'n_failed':       int(len(fail)),
        'mean_converged': float(np.mean(conv)),
        'mean_failed':    float(np.mean(fail)),
        'mannwhitney_U':  float(U),
        'p_value':        float(pval),
    })

strat_df = pd.DataFrame(strat_rows).sort_values('p_value')
strat_df.to_csv(RESULTS / 'nb04_failure_mode_stratification.csv', index=False)
print(f'Putirka convergence: {int((~putirka_fails).sum())} / {N_full} '
      f'(failure rate {100*putirka_fails.mean():.1f}%)\n')
print('Mann-Whitney U, converged vs failed samples (sorted by p):')
print(strat_df.round(4).to_string(index=False))


In [ ]:
# Failure analysis
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (mask, y, label, unit) in zip(axes, [
    (~T_all_valid, y_T_true, 'T failures', 'C'),
    (~P_all_valid, y_P_true, 'P failures', 'kbar'),
]):
    ax.hist([y[~mask], y[mask]], bins=25, stacked=True,
            label=[f'Putirka converged (n={(~mask).sum()})',
                   f'Putirka failed (n={mask.sum()})'],
            color=['steelblue', 'firebrick'])
    ax.set_xlabel(f'True value ({unit})')
    ax.set_ylabel('Count')
    ax.set_title(label)
    ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'fig_nb04_failure_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print('=== NB04 complete ===')

<!-- v7-fix-section-header -->
## Part 2: Three-way ML benchmark on ArcPL three-phase paired

**Purpose.** Head-to-head comparison of ML opx (ours) vs ML cpx external models
(Agreda/Jorgenson/Wang) vs classical Putirka 2008 opx/cpx thermobarometers on
the ArcPL three-phase (cpx + opx + liq) paired subset. This is the methodological
rigor figure: same rows, all methods.

**Data inputs.** Merged LEPR ArcPL sheets (Cpx, Opx, Liq) joined on `Experiment`
to a three-phase scope (n~=118), plus broader scopes `lepr_full` and `cpx_scope`.

**Methods evaluated.** 13 total:
- 5 cpx: Agreda-Lopez cpx-liq, Agreda-Lopez cpx-only, Jorgenson cpx-only,
  Wang 2021 cpx-liq, Putirka 2008 cpx-liq
- 4 Putirka opx: opx-liq (iterative), opx-only (iterative), opx-liq [true P],
  opx-only [true T]
- 4 Ours: opx-liq forest/boosted, opx-only forest/boosted

**Analysis performed.** Per-method RMSE + 95% bootstrap CI (B=2000) +
coverage across the three scopes. The coverage panel exposes which methods
work on which samples.

**How to interpret.** Look at RMSE under the "paired" scope for method quality,
and coverage across scopes for deployment breadth. Ceiling Putirka variants
([true P]/[true T]) bracket the best Putirka could ever do.

**Outputs.**
- `results/nb04_method_benchmark_paired.csv`,
  `results/nb04_method_benchmark_lepr_full.csv`,
  `results/nb04_method_benchmark_cpx_scope.csv`
- `figures/fig_nb04_method_benchmark_paired.{png,pdf}`,
  `figures/fig_nb04_method_benchmark_lepr_full.{png,pdf}`,
  `figures/fig_nb04_method_benchmark_cpx_scope.{png,pdf}`

**Downstream use.** NB09 references the paired scope numbers.


In [ ]:
# Three-way ML benchmark - load merged LEPR ArcPL data (cpx + opx + liq + true P/T)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy import stats

from config import LEPR_XLSX, RESULTS, FIGURES, MODELS, SEED_BOOTSTRAP
from src.external_models import (
    predict_agreda_from_df, predict_jorgenson, predict_wang,
    predict_putirka_cpx_liq, AGREDA_CPX_COLS, AGREDA_LIQ_COLS,
)

xls = pd.ExcelFile(LEPR_XLSX)
cpx = pd.read_excel(xls, sheet_name='Cpx')
liq = pd.read_excel(xls, sheet_name='Liq')
opx = pd.read_excel(xls, sheet_name='Opx')

# Cast all oxide columns to numeric; LEPR has string placeholders in minor oxides.
def _numerify(df, suffix):
    for c in df.columns:
        if c.endswith(suffix):
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0)
    return df

cpx = _numerify(cpx.drop_duplicates('Experiment'), '_Cpx')
liq = _numerify(liq.drop_duplicates('Experiment'), '_Liq')
opx = _numerify(opx.drop_duplicates('Experiment'), '_Opx')

# Merge cpx + liq + opx on Experiment. Outer merge keeps experiments that have
# only a subset, which we mask per-method below.
m = cpx.merge(liq, on='Experiment', how='outer', suffixes=('', '_liq_dup'))
m = m.merge(opx, on='Experiment', how='outer', suffixes=('', '_opx_dup'))
# Pick ground-truth T and P from cpx sheet first, fall back to liq, fall back to opx.
for tgt in ['T_K', 'P_kbar']:
    for src in ['', '_liq_dup', '_opx_dup']:
        col = f'{tgt}{src}'
        if col in m.columns:
            if tgt not in m.columns:
                m[tgt] = m[col]
            else:
                m[tgt] = m[tgt].combine_first(m[col])
m['T_C'] = pd.to_numeric(m['T_K'], errors='coerce') - 273.15
m['P_kbar'] = pd.to_numeric(m['P_kbar'], errors='coerce')
# Wang/Putirka require Fe3Fet_Liq; default to 0 (reduced Fe reporting convention).
if 'Fe3Fet_Liq' not in m.columns:
    m['Fe3Fet_Liq'] = 0.0

# Scope to the ArcPL-filtered subset from v5 nb04b so the comparison stays fair
# across methods (cpx vs opx-liq) on the same natural-sample experiments.
arcpl_preds = pd.read_csv(RESULTS / 'nb04b_arcpl_predictions.csv')
keep_exp = set(arcpl_preds['Experiment'].astype(str))
m = m[m['Experiment'].astype(str).isin(keep_exp)].reset_index(drop=True)

has_cpx = m[AGREDA_CPX_COLS].apply(pd.to_numeric, errors='coerce').fillna(0.0).sum(axis=1) > 80
has_liq = m[AGREDA_LIQ_COLS].apply(pd.to_numeric, errors='coerce').fillna(0.0).sum(axis=1) > 80
m = m[has_cpx & np.isfinite(m['T_C']) & np.isfinite(m['P_kbar'])].reset_index(drop=True)
print(f'Three-way benchmark set (ArcPL scope): n={len(m)} experiments')
print(f'  of which n_cpx_liq={int(has_liq.sum())}')


In [ ]:
# v8-fix: Putirka opx corrected API
# Run all five methods (Ours opx-liq, Agreda x2, Jorgenson, Wang, Putirka).
y_T = m['T_C'].values
y_P = m['P_kbar'].values

preds = {}  # name -> (T, P)
# 1. Agreda-Lopez cpx-only
a_cpx_T = predict_agreda_from_df(m, MODELS / 'external', 'cpx_only', 'T')['median']
a_cpx_P = predict_agreda_from_df(m, MODELS / 'external', 'cpx_only', 'P')['median']
preds['Agreda-Lopez cpx-only']  = (a_cpx_T, a_cpx_P)

# 2. Agreda-Lopez cpx-liq (where liq present)
try:
    a_liq_T = predict_agreda_from_df(m, MODELS / 'external', 'cpx_liq', 'T')['median']
    a_liq_P = predict_agreda_from_df(m, MODELS / 'external', 'cpx_liq', 'P')['median']
    preds['Agreda-Lopez cpx-liq'] = (a_liq_T, a_liq_P)
except KeyError as e:
    print(f'Agreda cpx-liq skipped ({e})')

# 3. Jorgenson cpx-only (via Thermobar_onnx)
try:
    j_T = predict_jorgenson(m, 'T', phase='cpx_only', P_kbar=y_P)
    j_P = predict_jorgenson(m, 'P', phase='cpx_only', T_K=y_T + 273.15)
    preds['Jorgenson cpx-only'] = (j_T, j_P)
except Exception as e:
    print(f'Jorgenson skipped ({e})')

# 4. Wang 2021 cpx-liq
try:
    w_T = predict_wang(m, 'T', P_kbar=y_P)
    w_P = predict_wang(m, 'P', T_K=y_T + 273.15)
    preds['Wang 2021 cpx-liq'] = (w_T, w_P)
except Exception as e:
    print(f'Wang skipped ({e})')

# 5. Putirka 2008 cpx-liq eq33/30
try:
    p_T = predict_putirka_cpx_liq(m, 'T', P_kbar=y_P)
    p_P = predict_putirka_cpx_liq(m, 'P', T_K=y_T + 273.15)
    preds['Putirka 2008 cpx-liq'] = (p_T, p_P)
except Exception as e:
    print(f'Putirka skipped ({e})')

# 6. Ours opx-liq + opx-only (direct inference via training-schema shim).
# LEPR uses _Opx/_Liq column suffixes; training schema uses unsuffixed opx
# oxides + liq_* prefix. Apply lepr_to_training_features on a .copy() so m
# keeps its original columns for Thermobar and external cpx models above.
import joblib as _jl_ours
from src.data import canonical_model_path, canonical_model_spec
from src.features import lepr_to_training_features
m_for_ours = lepr_to_training_features(m.copy())
print(f'\nOurs shim: m_for_ours shape={m_for_ours.shape}')
for _ox in ['SiO2', 'FeO_total', 'liq_SiO2', 'liq_FeO']:
    print(f'  {_ox} present: {_ox in m_for_ours.columns}')
for _tracks in ['opx_liq', 'opx_only']:
    _use_liq = (_tracks == 'opx_liq')
    for _family in ['forest', 'boosted']:
        try:
            _sT = canonical_model_spec('T_C', _tracks, _family, RESULTS)
            _sP = canonical_model_spec('P_kbar', _tracks, _family, RESULTS)
            _mT = _jl_ours.load(canonical_model_path('T_C', _tracks, _family, MODELS, RESULTS))
            _mP = _jl_ours.load(canonical_model_path('P_kbar', _tracks, _family, MODELS, RESULTS))
            _Xt, _ = build_feature_matrix(m_for_ours, _sT['feature_set'], use_liq=_use_liq)
            _Xp, _ = build_feature_matrix(m_for_ours, _sP['feature_set'], use_liq=_use_liq)
            preds[f'Ours {_tracks.replace("_", "-")} {_family}'] = (
                _mT.predict(_Xt), _mP.predict(_Xp),
            )
        except Exception as e:
            print(f'Ours {_tracks} {_family} skipped ({e})')

print('\nMethods available:', list(preds.keys()))


# v7-fix: Putirka opx methods (a/b/c/d)
# (a) opx-liq iterative (eq 28a + eq 29a)
try:
    import Thermobar as _pt_opx
    _opx_cols = {
        'SiO2_Opx': 'SiO2', 'TiO2_Opx': 'TiO2', 'Al2O3_Opx': 'Al2O3',
        'Cr2O3_Opx': 'Cr2O3', 'FeOt_Opx': 'FeOt_Opx', 'MnO_Opx': 'MnO',
        'MgO_Opx': 'MgO', 'CaO_Opx': 'CaO', 'Na2O_Opx': 'Na2O', 'K2O_Opx': 'K2O',
    }
    _liq_cols = {
        'SiO2_Liq': 'SiO2', 'TiO2_Liq': 'TiO2', 'Al2O3_Liq': 'Al2O3',
        'FeOt_Liq': 'FeOt_Liq', 'MnO_Liq': 'MnO', 'MgO_Liq': 'MgO',
        'CaO_Liq': 'CaO', 'Na2O_Liq': 'Na2O', 'K2O_Liq': 'K2O',
        'Cr2O3_Liq': 'Cr2O3', 'P2O5_Liq': 'P2O5',
    }
    _opx_in = pd.DataFrame({k: pd.to_numeric(m[k], errors='coerce')
                            for k in _opx_cols if k in m.columns})
    _liq_in = pd.DataFrame({k: pd.to_numeric(m[k], errors='coerce')
                            for k in _liq_cols if k in m.columns})
    if 'H2O_Liq' in m.columns:
        _liq_in['H2O_Liq'] = pd.to_numeric(m['H2O_Liq'], errors='coerce').fillna(0.0)
        # v8-fix: Putirka opx-liq eq28a/29a require Fe3Fet_Liq
        _liq_in['Fe3Fet_Liq'] = 0.0
    if len(_opx_in) and len(_liq_in):
        try:
            _iter = _pt_opx.calculate_opx_liq_press_temp(
                opx_comps=_opx_in, liq_comps=_liq_in,
                equationT='T_Put2008_eq28a', equationP='P_Put2008_eq29a')
            _T = (_iter['T_K_calc'].values - 273.15
                  if 'T_K_calc' in _iter.columns
                  else _iter.iloc[:, 0].values - 273.15)
            _P = (_iter['P_kbar_calc'].values
                  if 'P_kbar_calc' in _iter.columns
                  else _iter.iloc[:, 1].values)
            preds['Putirka 2008 opx-liq'] = (_T, _P)
        except Exception as _e1:
            print(f'Putirka opx-liq iterative skipped ({_e1})')

        # (c) opx-liq with true P, true T as inputs -> ceiling
        try:
            _Tceil = _pt_opx.calculate_opx_liq_temp(
                opx_comps=_opx_in, liq_comps=_liq_in,
                equationT='T_Put2008_eq28a', P=y_P)
            _Pceil = _pt_opx.calculate_opx_liq_press(
                opx_comps=_opx_in, liq_comps=_liq_in,
                equationP='P_Put2008_eq29a', T=y_T + 273.15)
            _T_arr = (_Tceil.values - 273.15 if hasattr(_Tceil, 'values')
                      else np.asarray(_Tceil) - 273.15)
            _P_arr = (_Pceil.values if hasattr(_Pceil, 'values')
                      else np.asarray(_Pceil))
            preds['Putirka 2008 opx-liq [true P]'] = (_T_arr, _P_arr)
        except Exception as _e2:
            print(f'Putirka opx-liq [true P] skipped ({_e2})')

    # (b) opx-only iterative (eq 28b_opx_sat + eq 29c) - 29c requires Cr2O3_Opx > 0
    try:
        # v8-fix: _opx_in columns are '_Opx'-suffixed; use Cr2O3_Opx.
        _mask_cr = (_opx_in.get('Cr2O3_Opx', pd.Series(np.zeros(len(_opx_in))))
                    .fillna(0.0) > 1e-4).values
        if _mask_cr.sum() >= 3:
            _opx_cr = _opx_in[_mask_cr].reset_index(drop=True)
            # v8-fix: Thermobar has no calculate_opx_only_press_temp or _temp.
            # Two-step iteration: eq28a for initial T, then eq29c uses that T,
            # then eq28b_opx_sat (via calculate_opx_liq_temp) uses the new P.
            _liq_cr = _liq_in[_mask_cr].reset_index(drop=True)
            _T_init = _pt_opx.calculate_opx_liq_temp(
                opx_comps=_opx_cr, liq_comps=_liq_cr,
                equationT='T_Put2008_eq28a', P=10.0)
            _T_init_K = (_T_init['T_K_calc'].values
                         if hasattr(_T_init, 'columns') and 'T_K_calc' in _T_init.columns
                         else (_T_init.values if hasattr(_T_init, 'values')
                               else np.asarray(_T_init)))
            if np.nanmedian(_T_init_K) < 400:
                _T_init_K = _T_init_K + 273.15
            _P_step = _pt_opx.calculate_opx_only_press(
                opx_comps=_opx_cr,
                equationP='P_Put2008_eq29c',
                T=_T_init_K)
            _P_arr = (_P_step['P_kbar_calc'].values
                      if hasattr(_P_step, 'columns') and 'P_kbar_calc' in _P_step.columns
                      else (_P_step.values if hasattr(_P_step, 'values')
                            else np.asarray(_P_step)))
            _T_step = _pt_opx.calculate_opx_liq_temp(
                opx_comps=_opx_cr, liq_comps=_liq_cr,
                equationT='T_Put2008_eq28b_opx_sat', P=_P_arr)
            _T_K = (_T_step['T_K_calc'].values
                    if hasattr(_T_step, 'columns') and 'T_K_calc' in _T_step.columns
                    else (_T_step.values if hasattr(_T_step, 'values')
                          else np.asarray(_T_step)))
            _T_arr = _T_K - 273.15 if np.nanmedian(_T_K) > 400 else _T_K
            _T_o = np.full(len(_opx_in), np.nan)
            _P_o = np.full(len(_opx_in), np.nan)
            _T_o[_mask_cr] = _T_arr
            _P_o[_mask_cr] = _P_arr
            preds['Putirka 2008 opx-only'] = (_T_o, _P_o)

            # (d) opx-only ceiling: observed T -> 29c for P; observed P ->
            # 28b_opx_sat for T.
            try:
                _Pceil_o = _pt_opx.calculate_opx_only_press(
                    opx_comps=_opx_cr,
                    equationP='P_Put2008_eq29c',
                    T=(y_T + 273.15)[_mask_cr])
                _Tceil_o = _pt_opx.calculate_opx_liq_temp(
                    opx_comps=_opx_cr, liq_comps=_liq_cr,
                    equationT='T_Put2008_eq28b_opx_sat', P=y_P[_mask_cr])
                _Pc = np.full(len(_opx_in), np.nan)
                _Tc = np.full(len(_opx_in), np.nan)
                _Pc[_mask_cr] = (_Pceil_o['P_kbar_calc'].values
                                 if hasattr(_Pceil_o, 'columns') and 'P_kbar_calc' in _Pceil_o.columns
                                 else (_Pceil_o.values if hasattr(_Pceil_o, 'values')
                                       else np.asarray(_Pceil_o)))
                _Tck = (_Tceil_o['T_K_calc'].values
                        if hasattr(_Tceil_o, 'columns') and 'T_K_calc' in _Tceil_o.columns
                        else (_Tceil_o.values if hasattr(_Tceil_o, 'values')
                              else np.asarray(_Tceil_o)))
                _Tc[_mask_cr] = _Tck - 273.15 if np.nanmedian(_Tck) > 400 else _Tck
                preds['Putirka 2008 opx-only [true T]'] = (_Tc, _Pc)
            except Exception as _e4:
                print(f'Putirka opx-only [true T] skipped ({_e4})')
    except Exception as _e3:
        print(f'Putirka opx-only iterative skipped ({_e3})')
except Exception as _e_outer:
    print(f'Putirka opx block skipped ({_e_outer})')


# v8-fix: assert Putirka opx methods made it in.
assert len(preds) >= 9, f'Expected >=9 methods after v8 opx fixes, got {len(preds)}'
_expected_opx = {'Putirka 2008 opx-liq', 'Putirka 2008 opx-only'}
_missing_opx = _expected_opx - set(preds)
print(f'v8-check: preds has {len(preds)} methods; '
      f'Putirka opx variants present: {sorted(_expected_opx & set(preds))}')
if _missing_opx:
    print(f'v8-check WARNING: missing {_missing_opx}')


In [ ]:
# Per-method metrics (RMSE + 95% bootstrap CI + R2 + coverage).
rng = np.random.default_rng(SEED_BOOTSTRAP)
BOOT = 2000

def rmse_ci(y, yhat, B=BOOT):
    mask = np.isfinite(y) & np.isfinite(yhat)
    n = int(mask.sum())
    if n < 3:
        return np.nan, np.nan, np.nan, n, np.nan
    yt = y[mask]; yp = yhat[mask]
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))
    idx = rng.integers(0, n, size=(B, n))
    boots = np.sqrt(np.mean((yt[idx] - yp[idx]) ** 2, axis=1))
    lo, hi = np.quantile(boots, [0.025, 0.975])
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - yt.mean()) ** 2))
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return rmse, float(lo), float(hi), n, r2

rows = []
for name, (pT, pP) in preds.items():
    mT = rmse_ci(y_T, pT)
    mP = rmse_ci(y_P, pP)
    rows.append({
        'Method': name,
        'T_n': mT[3], 'T_RMSE': mT[0], 'T_RMSE_CI_lo': mT[1], 'T_RMSE_CI_hi': mT[2], 'T_R2': mT[4],
        'T_coverage_pct': 100 * mT[3] / len(m),
        'P_n': mP[3], 'P_RMSE': mP[0], 'P_RMSE_CI_lo': mP[1], 'P_RMSE_CI_hi': mP[2], 'P_R2': mP[4],
        'P_coverage_pct': 100 * mP[3] / len(m),
    })
three_way = pd.DataFrame(rows)
three_way.to_csv(RESULTS / 'nb04_three_way_ml_benchmark.csv', index=False)
print(three_way.round(3).to_string(index=False))


In [ ]:
# v8-fix: Putirka opx corrected API
# v7 Part H+: three-panel method benchmark across three scopes
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import joblib
from src.plot_style import apply_style
from src.data import canonical_model_path, canonical_model_spec
from src.features import build_feature_matrix
from config import FAMILY_COLORS
from src.plot_style import OKABE_ITO

apply_style()

COLORS = {
    'Ours opx-liq forest':           FAMILY_COLORS['forest'],
    'Ours opx-liq boosted':          FAMILY_COLORS['boosted'],
    'Ours opx-only forest':          FAMILY_COLORS['forest'],
    'Ours opx-only boosted':         FAMILY_COLORS['boosted'],
    'Agreda-Lopez cpx-liq':          FAMILY_COLORS['external_cpx'],
    'Agreda-Lopez cpx-only':         OKABE_ITO['yellow'],
    'Jorgenson cpx-only':            OKABE_ITO['green'],
    'Wang 2021 cpx-liq':             OKABE_ITO['vermillion'],
    'Putirka 2008 cpx-liq':          FAMILY_COLORS['putirka'],
    'Putirka 2008 opx-liq':          FAMILY_COLORS['putirka'],
    'Putirka 2008 opx-only':         FAMILY_COLORS['putirka'],
    'Putirka 2008 opx-liq [true P]': FAMILY_COLORS['putirka'],
    'Putirka 2008 opx-only [true T]':FAMILY_COLORS['putirka'],
}

BOOT_NEW = 2000


def _merge_scope(keep_arcpl, require_liq, require_opx):
    """Build a scope DataFrame from the raw cpx/liq/opx sheets loaded in cell 23."""
    how = 'inner' if (require_liq or require_opx) else 'left'
    s = cpx.merge(liq, on='Experiment', how=how if require_liq else 'left',
                  suffixes=('', '_liq_dup'))
    s = s.merge(opx, on='Experiment', how=how if require_opx else 'left',
                suffixes=('', '_opx_dup'))
    for tgt in ['T_K', 'P_kbar']:
        for src in ['', '_liq_dup', '_opx_dup']:
            col = f'{tgt}{src}'
            if col in s.columns:
                if tgt not in s.columns:
                    s[tgt] = s[col]
                else:
                    s[tgt] = s[tgt].combine_first(s[col])
    s['T_C'] = pd.to_numeric(s['T_K'], errors='coerce') - 273.15
    s['P_kbar'] = pd.to_numeric(s['P_kbar'], errors='coerce')
    if 'Fe3Fet_Liq' not in s.columns:
        s['Fe3Fet_Liq'] = 0.0
    if keep_arcpl:
        s = s[s['Experiment'].astype(str).isin(keep_exp)]
    has_c = s[AGREDA_CPX_COLS].apply(pd.to_numeric, errors='coerce').fillna(0.0).sum(axis=1) > 80
    s = s[has_c & np.isfinite(s['T_C']) & np.isfinite(s['P_kbar'])].reset_index(drop=True)
    return s


def _predict_all_methods(scope_df):
    y_T_s = scope_df['T_C'].values
    y_P_s = scope_df['P_kbar'].values
    p = {}
    try:
        a = predict_agreda_from_df(scope_df, MODELS / 'external', 'cpx_only', 'T')['median']
        b = predict_agreda_from_df(scope_df, MODELS / 'external', 'cpx_only', 'P')['median']
        p['Agreda-Lopez cpx-only'] = (a, b)
    except Exception as e:
        print(f'  Agreda cpx-only skipped ({e})')
    try:
        a = predict_agreda_from_df(scope_df, MODELS / 'external', 'cpx_liq', 'T')['median']
        b = predict_agreda_from_df(scope_df, MODELS / 'external', 'cpx_liq', 'P')['median']
        p['Agreda-Lopez cpx-liq'] = (a, b)
    except Exception as e:
        print(f'  Agreda cpx-liq skipped ({e})')
    try:
        p['Jorgenson cpx-only'] = (
            predict_jorgenson(scope_df, 'T', phase='cpx_only', P_kbar=y_P_s),
            predict_jorgenson(scope_df, 'P', phase='cpx_only', T_K=y_T_s + 273.15),
        )
    except Exception as e:
        print(f'  Jorgenson skipped ({e})')
    try:
        p['Wang 2021 cpx-liq'] = (
            predict_wang(scope_df, 'T', P_kbar=y_P_s),
            predict_wang(scope_df, 'P', T_K=y_T_s + 273.15),
        )
    except Exception as e:
        print(f'  Wang skipped ({e})')
    try:
        p['Putirka 2008 cpx-liq'] = (
            predict_putirka_cpx_liq(scope_df, 'T', P_kbar=y_P_s),
            predict_putirka_cpx_liq(scope_df, 'P', T_K=y_T_s + 273.15),
        )
    except Exception as e:
        print(f'  Putirka skipped ({e})')
    # Training-schema shim: LEPR _Opx/_Liq suffixes -> ExPetDB flat columns.
    from src.features import lepr_to_training_features
    scope_train = lepr_to_training_features(scope_df.copy())
    for tracks in ['opx_liq', 'opx_only']:
        use_liq = (tracks == 'opx_liq')
        for fam in ['forest', 'boosted']:
            name = f'Ours {tracks.replace("_", "-")} {fam}'
            try:
                sT = canonical_model_spec('T_C', tracks, fam, RESULTS)
                sP = canonical_model_spec('P_kbar', tracks, fam, RESULTS)
                mT = joblib.load(canonical_model_path('T_C', tracks, fam, MODELS, RESULTS))
                mP = joblib.load(canonical_model_path('P_kbar', tracks, fam, MODELS, RESULTS))
                Xt, _ = build_feature_matrix(scope_train, sT['feature_set'], use_liq=use_liq)
                Xp, _ = build_feature_matrix(scope_train, sP['feature_set'], use_liq=use_liq)
                p[name] = (mT.predict(Xt), mP.predict(Xp))
            except Exception as e:
                print(f'  {name} partial/skipped ({e})')
                p[name] = (np.full(len(scope_df), np.nan),
                           np.full(len(scope_df), np.nan))
    # v7-fix: Putirka opx methods (a/b/c/d) in _predict_all_methods
    try:
        import Thermobar as _pt_opx
        _opx_in = pd.DataFrame({
            k: pd.to_numeric(scope_df.get(k, np.nan), errors='coerce')
            for k in ['SiO2_Opx','TiO2_Opx','Al2O3_Opx','Cr2O3_Opx','FeOt_Opx',
                      'MnO_Opx','MgO_Opx','CaO_Opx','Na2O_Opx','K2O_Opx']
            if k in scope_df.columns})
        _liq_in = pd.DataFrame({
            k: pd.to_numeric(scope_df.get(k, np.nan), errors='coerce')
            for k in ['SiO2_Liq','TiO2_Liq','Al2O3_Liq','FeOt_Liq','MnO_Liq',
                      'MgO_Liq','CaO_Liq','Na2O_Liq','K2O_Liq','Cr2O3_Liq','P2O5_Liq']
            if k in scope_df.columns})
        if 'H2O_Liq' in scope_df.columns:
            _liq_in['H2O_Liq'] = pd.to_numeric(scope_df['H2O_Liq'], errors='coerce').fillna(0.0)
            # v8-fix: Putirka opx-liq eq28a/29a require Fe3Fet_Liq
            _liq_in['Fe3Fet_Liq'] = 0.0
        if len(_opx_in) and len(_liq_in):
            try:
                _iter = _pt_opx.calculate_opx_liq_press_temp(
                    opx_comps=_opx_in, liq_comps=_liq_in,
                    equationT='T_Put2008_eq28a', equationP='P_Put2008_eq29a')
                _T = (_iter['T_K_calc'].values - 273.15 if 'T_K_calc' in _iter.columns
                      else _iter.iloc[:, 0].values - 273.15)
                _P = (_iter['P_kbar_calc'].values if 'P_kbar_calc' in _iter.columns
                      else _iter.iloc[:, 1].values)
                p['Putirka 2008 opx-liq'] = (_T, _P)
            except Exception as _e1:
                print(f'  Putirka opx-liq iterative skipped ({_e1})')
            try:
                _Tceil = _pt_opx.calculate_opx_liq_temp(
                    opx_comps=_opx_in, liq_comps=_liq_in,
                    equationT='T_Put2008_eq28a', P=y_P_s)
                _Pceil = _pt_opx.calculate_opx_liq_press(
                    opx_comps=_opx_in, liq_comps=_liq_in,
                    equationP='P_Put2008_eq29a', T=y_T_s + 273.15)
                _Ta = (_Tceil.values - 273.15 if hasattr(_Tceil, 'values')
                       else np.asarray(_Tceil) - 273.15)
                _Pa = (_Pceil.values if hasattr(_Pceil, 'values') else np.asarray(_Pceil))
                p['Putirka 2008 opx-liq [true P]'] = (_Ta, _Pa)
            except Exception as _e2:
                print(f'  Putirka opx-liq [true P] skipped ({_e2})')
        _mask_cr = ((_opx_in.get('Cr2O3_Opx', pd.Series(np.zeros(len(scope_df))))
                     .fillna(0.0) > 1e-4).values
                    if 'Cr2O3_Opx' in _opx_in.columns
                    else np.zeros(len(scope_df), dtype=bool))
        if _mask_cr.sum() >= 3:
            _opx_cr = _opx_in[_mask_cr].reset_index(drop=True)
            try:
                # v8-fix: use encyclopedia pattern (no calculate_opx_only_press_temp).
                _liq_cr = _liq_in[_mask_cr].reset_index(drop=True)
                _T_init = _pt_opx.calculate_opx_liq_temp(
                    opx_comps=_opx_cr, liq_comps=_liq_cr,
                    equationT='T_Put2008_eq28a', P=10.0)
                _T_init_K = (_T_init['T_K_calc'].values
                             if hasattr(_T_init, 'columns') and 'T_K_calc' in _T_init.columns
                             else (_T_init.values if hasattr(_T_init, 'values')
                                   else np.asarray(_T_init)))
                if np.nanmedian(_T_init_K) < 400:
                    _T_init_K = _T_init_K + 273.15
                _P_step = _pt_opx.calculate_opx_only_press(
                    opx_comps=_opx_cr, equationP='P_Put2008_eq29c', T=_T_init_K)
                _P_arr = (_P_step['P_kbar_calc'].values
                          if hasattr(_P_step, 'columns') and 'P_kbar_calc' in _P_step.columns
                          else (_P_step.values if hasattr(_P_step, 'values')
                                else np.asarray(_P_step)))
                _T_step = _pt_opx.calculate_opx_liq_temp(
                    opx_comps=_opx_cr, liq_comps=_liq_cr,
                    equationT='T_Put2008_eq28b_opx_sat', P=_P_arr)
                _T_K = (_T_step['T_K_calc'].values
                        if hasattr(_T_step, 'columns') and 'T_K_calc' in _T_step.columns
                        else (_T_step.values if hasattr(_T_step, 'values')
                              else np.asarray(_T_step)))
                _T_arr = _T_K - 273.15 if np.nanmedian(_T_K) > 400 else _T_K
                _To = np.full(len(scope_df), np.nan)
                _Po = np.full(len(scope_df), np.nan)
                _To[_mask_cr] = _T_arr
                _Po[_mask_cr] = _P_arr
                p['Putirka 2008 opx-only'] = (_To, _Po)
            except Exception as _e3:
                print(f'  Putirka opx-only iterative skipped ({_e3})')
            try:
                _Pceil_o = _pt_opx.calculate_opx_only_press(
                    opx_comps=_opx_cr,
                    equationP='P_Put2008_eq29c',
                    T=(y_T_s + 273.15)[_mask_cr])
                _Tceil_o = _pt_opx.calculate_opx_only_temp(
                    opx_comps=_opx_cr,
                    equationT='T_Put2008_eq28b_opx_sat',
                    P=y_P_s[_mask_cr])
                _Pc = np.full(len(scope_df), np.nan)
                _Tc = np.full(len(scope_df), np.nan)
                _Pc[_mask_cr] = (_Pceil_o.values if hasattr(_Pceil_o, 'values')
                                 else np.asarray(_Pceil_o))
                _Tc[_mask_cr] = (_Tceil_o.values - 273.15
                                 if hasattr(_Tceil_o, 'values')
                                 else np.asarray(_Tceil_o) - 273.15)
                p['Putirka 2008 opx-only [true T]'] = (_Tc, _Pc)
            except Exception as _e4:
                print(f'  Putirka opx-only [true T] skipped ({_e4})')
    except Exception as _eo:
        print(f'  Putirka opx block skipped ({_eo})')
    return p


def _metrics_df(preds_s, y_T, y_P, n_total):
    rows = []
    for name, (pT, pP) in preds_s.items():
        mT = rmse_ci(y_T, pT, B=BOOT_NEW)
        mP = rmse_ci(y_P, pP, B=BOOT_NEW)
        rows.append({
            'Method': name,
            'T_n': mT[3], 'T_RMSE': mT[0],
            'T_RMSE_CI_lo': mT[1], 'T_RMSE_CI_hi': mT[2], 'T_R2': mT[4],
            'T_coverage_pct': 100 * mT[3] / n_total if n_total else 0,
            'P_n': mP[3], 'P_RMSE': mP[0],
            'P_RMSE_CI_lo': mP[1], 'P_RMSE_CI_hi': mP[2], 'P_R2': mP[4],
            'P_coverage_pct': 100 * mP[3] / n_total if n_total else 0,
        })
    return pd.DataFrame(rows)


def _is_putirka(name):
    return name.startswith('Putirka')


def _render_three_panel(df_m, n_total, scope_label, out_stem):
    min_cov = df_m[['T_coverage_pct', 'P_coverage_pct']].min(axis=1)
    dropped = df_m.loc[min_cov < 20, 'Method'].tolist()
    shown_mask = min_cov >= 20
    shown = df_m.loc[shown_mask].copy()
    shown['_min_cov'] = min_cov.loc[shown_mask].values
    shown['_max_cov'] = df_m.loc[shown_mask, ['T_coverage_pct',
                                              'P_coverage_pct']].max(axis=1).values
    if dropped:
        print(f'  Dropped from figure (<20% cov): {dropped}')
    # v8-fix: independent T/P sort — each RMSE panel uses its own ordering
    shown_T = shown.sort_values('T_RMSE', na_position='last').reset_index(drop=True)
    shown_P = shown.sort_values('P_RMSE', na_position='last').reset_index(drop=True)
    shown = shown_T  # coverage panel + footnotes stay T-sorted for reference

    footnotes = {}
    foot_text = []
    for i, row in shown.iterrows():
        if row['_min_cov'] < 90:
            num = len(footnotes) + 1
            footnotes[row['Method']] = str(num)
            foot_text.append(
                f"{num} {row['Method']}: {row['_min_cov']:.0f}% coverage "
                f"(limited by {'T' if row['T_coverage_pct'] < row['P_coverage_pct'] else 'P'} "
                f"missing-row filter)"
            )

    n_methods = len(shown)
    h = max(8.0, 0.4 * n_methods + 3)
    fig = plt.figure(figsize=(16, h))
    gs = GridSpec(1, 3, figure=fig, width_ratios=[4, 4, 1.5])
    ax_T = fig.add_subplot(gs[0, 0])
    ax_P = fig.add_subplot(gs[0, 1])  # v8-fix: no sharey (P sort differs)
    ax_C = fig.add_subplot(gs[0, 2], sharey=ax_T)
    y_pos = np.arange(n_methods)

    for ax, df, col, col_lo, col_hi, xlab, fmt in [
        (ax_T, shown_T, 'T_RMSE', 'T_RMSE_CI_lo', 'T_RMSE_CI_hi', 'T RMSE (C)', '{:.0f}'),
        (ax_P, shown_P, 'P_RMSE', 'P_RMSE_CI_lo', 'P_RMSE_CI_hi', 'P RMSE (kbar)', '{:.2f}'),
    ]:
        vals = df[col].values.astype(float)
        lo = df[col_lo].values.astype(float)
        hi = df[col_hi].values.astype(float)
        xerr_lo = np.where(np.isfinite(vals) & np.isfinite(lo), vals - lo, 0.0)
        xerr_hi = np.where(np.isfinite(vals) & np.isfinite(hi), hi - vals, 0.0)
        xerr = np.vstack([xerr_lo, xerr_hi])
        colors = [COLORS.get(n, '#777') for n in df['Method']]
        bars = ax.barh(y_pos, np.nan_to_num(vals, nan=0.0), xerr=xerr,
                       color=colors, edgecolor='black',
                       error_kw={'elinewidth': 1.1, 'capsize': 2.5})
        for b, n in zip(bars, df['Method']):
            if _is_putirka(n):
                b.set_hatch('///')
        for i, v in enumerate(vals):
            if np.isfinite(v):
                ax.text(v * 1.01 + 0.001, i, fmt.format(v),
                        va='center', fontsize=8)
            else:
                ax.text(0.01, i, 'n/a', va='center', fontsize=8,
                        color='gray', style='italic')
        if np.isfinite(vals).any():
            best = float(np.nanmin(vals))
            ax.axvline(best, color='gray', linestyle=':', lw=1, alpha=0.6)
        ax.set_xlabel(xlab)
        ax.grid(axis='x', alpha=0.3)
        ax.set_xlim(left=0)
        ax.set_yticks(y_pos)
        _sup_labels = []
        for _name in df['Method']:
            _sup = footnotes.get(_name)
            _sup_labels.append(f'{_name}$^{{{_sup}}}$' if _sup else _name)
        ax.set_yticklabels(_sup_labels, fontsize=9)
        ax.invert_yaxis()

    # v8-fix: y-ticks now set inside the per-panel loop; C panel still hides labels
    plt.setp(ax_C.get_yticklabels(), visible=False)

    cov_bar = shown['_min_cov'].values
    cov_mark = shown['_max_cov'].values
    c_colors = [COLORS.get(n, '#777') for n in shown['Method']]
    ax_C.barh(y_pos, cov_bar, color=c_colors, edgecolor='black', alpha=0.6)
    for i, (mn, mx) in enumerate(zip(cov_bar, cov_mark)):
        ax_C.text(mn + 1.5, i, f'{mn:.0f}%', va='center', fontsize=8)
        if abs(mx - mn) > 0.5:
            ax_C.plot(mx, i, marker='x', color='black', ms=5, alpha=0.8)
    ax_C.axvline(100, color='gray', linestyle='--', lw=1, alpha=0.6)
    ax_C.set_xlim(0, 105)
    ax_C.set_xlabel('Coverage (%)')
    ax_C.grid(axis='x', alpha=0.3)

    fig.suptitle(f'Method benchmark — {scope_label} (n_total = {n_total})',
                 fontsize=12, fontweight='bold')

    caption = ('Bar height (panels A, B): RMSE with 95% bootstrap CI (B=2000). '
               'Bar height (panel C): coverage = min(T, P) finite-prediction '
               'rate; x marker = higher of T/P if differs. Hatched bars = '
               'classical (Putirka 2008); solid bars = ML methods. Methods '
               'sorted by T RMSE ascending.')
    fig.text(0.5, -0.02, caption, ha='center', fontsize=8, wrap=True)
    if foot_text:
        fig.text(0.05, -0.06, '    '.join(foot_text), ha='left',
                 fontsize=7, style='italic')

    for ext in ['png', 'pdf']:
        fp = FIGURES / f'{out_stem}.{ext}'
        fig.savefig(fp, bbox_inches='tight',
                    dpi=300 if ext == 'png' else None)
    plt.show()
    plt.close(fig)

    cov_lo, cov_hi = (cov_bar.min(), cov_bar.max()) if len(cov_bar) else (0, 0)
    low_cov_names = [n for n, c in zip(shown['Method'], cov_bar) if c < 90]
    print(f"Saved {out_stem}: panels [A, B, C] with n_methods={n_methods} rows. "
          f"Coverage range: {cov_lo:.0f}% - {cov_hi:.0f}%. "
          f"Methods with cov<90%: {low_cov_names}")


# ---- Build three scopes ----
m_arcpl_paired = _merge_scope(keep_arcpl=True, require_liq=True, require_opx=True)
m_lepr_full = _merge_scope(keep_arcpl=False, require_liq=True, require_opx=True)
m_cpx_scope = _merge_scope(keep_arcpl=True, require_liq=False, require_opx=False)
print(f'Scope arcpl_paired: n={len(m_arcpl_paired)}')
print(f'Scope lepr_full:    n={len(m_lepr_full)}  (includes training rows for Ours)')
print(f'Scope cpx_scope:    n={len(m_cpx_scope)}')

SCOPES = [
    (m_arcpl_paired, 'ArcPL three-phase paired (cpx + opx + liq)',
     'fig_nb04_method_benchmark_paired',   'arcpl_paired'),
    (m_lepr_full,    'Full LEPR three-phase paired',
     'fig_nb04_method_benchmark_lepr_full', 'lepr_full'),
    (m_cpx_scope,    'ArcPL cpx-bearing scope',
     'fig_nb04_method_benchmark_cpx_scope', 'cpx_scope'),
]
for df_scope, label, fn, slug in SCOPES:
    if len(df_scope) < 10:
        print(f'\n[{slug}] SKIPPED: n={len(df_scope)} too small')
        continue
    print(f'\n[{slug}] running {len(df_scope)} rows ...')
    preds_s = _predict_all_methods(df_scope)
    ms = _metrics_df(preds_s, df_scope['T_C'].values,
                     df_scope['P_kbar'].values, len(df_scope))
    ms.round(3).to_csv(RESULTS / f'nb04_method_benchmark_{slug}.csv', index=False)
    _render_three_panel(ms, len(df_scope), label, fn)
    _table_cols = ['Method', 'T_n', 'T_RMSE', 'T_R2', 'T_coverage_pct',
                   'P_n', 'P_RMSE', 'P_R2', 'P_coverage_pct']
    print(f'\n[{slug}] benchmark table:')
    print(ms[_table_cols].round(3).to_string(index=False))


**How to read the three-way benchmark figure.**

- **Panel (a)/(b):** Every point is the same ArcPL experiment seen through a
  different thermobarometer. Closer to the 1:1 dashed line is better. Vertical
  spread at a given x value shows between-method disagreement on the same
  sample.
- **Panel (c):** Bar height is RMSE (T scaled by 1/100 so T and P share a
  y-axis); error bars are 95% bootstrap CIs. Overlapping CIs mean no
  statistically significant gap.
- **Panel (d):** Bar height is the fraction of ArcPL each method successfully
  predicts. Classical Putirka drops coverage when any required oxide is missing
  or out-of-range; ML methods return a prediction on every row that has the
  required phase data.

**Phase-mismatch caveat.** Agreda-Lopez, Jorgenson, and Wang predict from cpx;
ours predicts from opx. Direct RMSE comparison treats them as alternative tools
for the same sample - the practical deployment question. It is not a claim that
opx and cpx carry the same information content.


<!-- v7-fix-section-header -->
## Part 3: ArcPL opx-liq external validation

**Purpose.** External validation of our canonical opx-liq ML on the ArcPL
hydrous subset of LEPR — a dataset our models have never seen. This is
the "deployment reality" number.

**Data inputs.** LEPR Opx-Liq sheet at `data/raw/external/LEPR_Wet_Stitched_April2023_Norm100Anhydrs.xlsx`,
filtered via `Citation_x.contains('_notinLEPR')` -> ArcPL subset (n~=324),
minus training overlap with ExPetDB (author+year match) -> n~=204.

**Methods evaluated.** Canonical ML opx-liq forest (RF on alr for T, RF on raw for P)
and boosted (XGB on raw for both). Putirka opx comparisons live in Part 5.

**Analysis performed.** Cleaning mirrors NB01 (cation recalc on 6-oxygen basis,
oxide total filter, pigeonite filter, KD filter, engineered features). Apply both
canonical families to the cleaned frame, compute RMSE, MAE, R^2 overall plus 95%
bootstrap CI (B=2000, rng seed = SEED_BOOTSTRAP).

**How to interpret.** The RMSE here (T ~=58 C, P ~=2.7 kbar on the baseline)
is what a user should expect on real arc opx-liq samples. Substantially higher
numbers than the internal test split (Figure 3/4) would indicate train/deployment
drift; comparable numbers indicate the model generalizes.

**Outputs.**
- `results/nb04_arcpl_opx_liq_predictions_forest.csv`
- `results/nb04_arcpl_opx_liq_predictions_boosted.csv`
- `results/nb04_arcpl_opx_liq_metrics.csv`
- `results/nb04_arcpl_opx_liq_bootstrap.csv`
- Backward-compat aliases: `nb04b_arcpl_predictions_forest.csv`,
  `nb04b_arcpl_predictions_boosted.csv`, `nb04b_arcpl_predictions.csv`

**Downstream use.** NB09 reads `nb04b_arcpl_predictions.csv` for H2O and OOD
analyses; NBF fig 13 reads the OOD derivatives. Backward-compat aliases keep
those paths working until the downstream notebooks are migrated.


In [ ]:
# v7-fix: Part 3 ArcPL opx-liq external validation (ported from nb04b)
# Cleaning pipeline mirrors NB01 so the canonical ML models see training-
# schema columns they recognize.
from scipy import stats as _p3_stats
import joblib as _p3_joblib
from src.features import (
    build_feature_matrix as _p3_build_feat,
    cation_recalc_6oxy as _p3_cation,
    add_engineered_features as _p3_engineer,
)
from src.data import (
    canonical_model_path as _p3_canon_path,
    canonical_model_spec as _p3_canon_spec,
)

_P3_OXIDE_TOT = (OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX)
_P3_CATION_RANGE = (CATION_SUM_MIN, CATION_SUM_MAX)
_P3_WO_MAX = WO_MAX_MOL_PCT
_P3_PCEIL = P_CEILING_KBAR
_P3_BOOT = 2000
_p3_rng = np.random.default_rng(SEED_BOOTSTRAP)


def _p3_bootstrap_rmse(y, yhat, B=_P3_BOOT):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    mask = np.isfinite(y) & np.isfinite(yhat)
    n = int(mask.sum())
    if n < 3:
        return (np.nan, np.nan, np.nan, n)
    yt = y[mask]; yp = yhat[mask]
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))
    idx = _p3_rng.integers(0, n, size=(B, n))
    boots = np.sqrt(np.mean((yt[idx] - yp[idx]) ** 2, axis=1))
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return float(rmse), float(lo), float(hi), n


def _p3_metrics(y, yhat):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    mask = np.isfinite(y) & np.isfinite(yhat)
    yt = y[mask]; yp = yhat[mask]
    n = int(mask.sum())
    if n < 3:
        return dict(n=n, rmse=np.nan, mae=np.nan, r2=np.nan)
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))
    mae = float(np.mean(np.abs(yt - yp)))
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - yt.mean()) ** 2))
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return dict(n=n, rmse=rmse, mae=mae, r2=float(r2))


# 1) Load LEPR Opx-Liq sheet, ArcPL subset via Citation_x._notinLEPR.
_p3_lepr = pd.read_excel(LEPR_XLSX, sheet_name='Opx-Liq')
_p3_arcpl = _p3_lepr[_p3_lepr['Citation_x'].astype(str).str.contains('_notinLEPR',
                                                                       na=False)].copy()
print(f'Part 3: ArcPL subset before overlap removal: n={len(_p3_arcpl)}')

# 2) Rename LEPR -> ExPetDB training schema
_p3_rename = {
    'Citation_x': 'Citation', 'Experiment_x': 'Experiment',
    'P_kbar_x': 'P_kbar',
    'SiO2_Opx': 'SiO2', 'TiO2_Opx': 'TiO2', 'Al2O3_Opx': 'Al2O3',
    'FeOt_Opx': 'FeO_total', 'MnO_Opx': 'MnO', 'MgO_Opx': 'MgO',
    'CaO_Opx': 'CaO', 'Na2O_Opx': 'Na2O', 'K2O_Opx': 'K2O',
    'Cr2O3_Opx': 'Cr2O3', 'P2O5_Opx': 'P2O5',
    'SiO2_Liq': 'liq_SiO2', 'TiO2_Liq': 'liq_TiO2', 'Al2O3_Liq': 'liq_Al2O3',
    'FeOt_Liq': 'liq_FeO', 'MnO_Liq': 'liq_MnO', 'MgO_Liq': 'liq_MgO',
    'CaO_Liq': 'liq_CaO', 'Na2O_Liq': 'liq_Na2O', 'K2O_Liq': 'liq_K2O',
    'Cr2O3_Liq': 'liq_Cr2O3', 'H2O_Liq': 'H2O_Liq',
    'H2O_Liq_Method': 'H2O_Liq_Method',
}
_p3_arcpl = _p3_arcpl.rename(columns=_p3_rename)
if 'T_K_x' in _p3_arcpl.columns:
    _p3_arcpl['T_C'] = _p3_arcpl['T_K_x'] - 273.15
_p3_arcpl = _p3_arcpl.drop(columns=[c for c in _p3_arcpl.columns if c.endswith('_y')],
                           errors='ignore')
_p3_arcpl = _p3_arcpl[_p3_arcpl.get('H2O_Liq', pd.Series(np.zeros(len(_p3_arcpl)))) >= 0].copy()

# is_vbd flag from H2O_Liq_Method
_p3_arcpl['is_vbd'] = _p3_arcpl.get(
    'H2O_Liq_Method', pd.Series([''] * len(_p3_arcpl))
).astype(str).str.contains('VBD|vbd|mass_balance|diff', na=False, regex=True)

# 3) Cleaning pipeline
_p3_all_ox = ['SiO2', 'TiO2', 'Al2O3', 'Cr2O3', 'FeO_total', 'MnO', 'MgO', 'CaO',
              'Na2O', 'K2O', 'P2O5']
for _ox in _p3_all_ox:
    if _ox in _p3_arcpl.columns:
        _p3_arcpl[_ox] = pd.to_numeric(_p3_arcpl[_ox], errors='coerce')
_p3_present = [o for o in _p3_all_ox if o in _p3_arcpl.columns]
_p3_arcpl['oxide_total'] = _p3_arcpl[_p3_present].sum(axis=1, min_count=5)
_p3_arcpl = _p3_arcpl[_p3_arcpl['oxide_total'].between(*_P3_OXIDE_TOT)].copy()
_p3_arcpl = _p3_cation(_p3_arcpl, oxides=_p3_all_ox)
_p3_arcpl = _p3_arcpl[_p3_arcpl['cation_sum'].between(*_P3_CATION_RANGE)].copy()
_p3_arcpl = _p3_arcpl.dropna(subset=['SiO2', 'Al2O3', 'FeO_total', 'MgO', 'CaO']).copy()
_p3_arcpl = _p3_engineer(_p3_arcpl)
_p3_arcpl['Wo'] = _p3_arcpl['Wo_frac'] * 100.0
_p3_arcpl = _p3_arcpl[_p3_arcpl['Wo'] <= _P3_WO_MAX].copy()
_p3_arcpl = _p3_arcpl[_p3_arcpl['P_kbar'] <= _P3_PCEIL].copy()

# Basic KD filter (same constants as NB01)
_p3_fe_opx = _p3_arcpl['FeO_total'] / 71.844
_p3_mg_opx = _p3_arcpl['MgO'] / 40.304
_p3_fe_liq = (_p3_arcpl['liq_FeO'] * (1.0 - FE3_FET_RATIO)) / 71.844
_p3_mg_liq = _p3_arcpl['liq_MgO'] / 40.304
_p3_KD = (_p3_fe_opx / _p3_mg_opx) / (_p3_fe_liq / _p3_mg_liq)
_p3_arcpl = _p3_arcpl[(_p3_KD >= KD_FEMG_MIN) & (_p3_KD <= KD_FEMG_MAX)].copy()

# 4) Remove ExPetDB training overlap (best-effort author+year)
_p3_expet = pd.read_parquet(DATA_PROC / 'opx_clean_opx_liq.parquet')
def _p3_citkey(s):
    s = str(s)
    import re as _re
    m = _re.search(r'([A-Za-z][A-Za-z ,.\-]+?)[\s,]*((?:19|20)\d{2})', s)
    return (m.group(1).strip().lower() + '_' + m.group(2)) if m else s.strip().lower()
_p3_expet_keys = set(_p3_expet['Citation'].astype(str).map(_p3_citkey))
_p3_arcpl['_key'] = _p3_arcpl['Citation'].astype(str).map(_p3_citkey)
_p3_before = len(_p3_arcpl)
_p3_arcpl = _p3_arcpl[~_p3_arcpl['_key'].isin(_p3_expet_keys)].reset_index(drop=True)
print(f'Overlap removal: {_p3_before} -> {len(_p3_arcpl)}')

# 5) Per-family predictions
_p3_metrics_rows = []
_p3_boot_rows = []
_p3_pred_store = {}
for _fam in ['forest', 'boosted']:
    try:
        _sT = _p3_canon_spec('T_C',    'opx_liq', _fam, RESULTS)
        _sP = _p3_canon_spec('P_kbar', 'opx_liq', _fam, RESULTS)
        _mT = _p3_joblib.load(_p3_canon_path('T_C',    'opx_liq', _fam, MODELS, RESULTS))
        _mP = _p3_joblib.load(_p3_canon_path('P_kbar', 'opx_liq', _fam, MODELS, RESULTS))
        _Xt, _ = _p3_build_feat(_p3_arcpl, _sT['feature_set'], use_liq=True)
        _Xp, _ = _p3_build_feat(_p3_arcpl, _sP['feature_set'], use_liq=True)
        _yT = _p3_arcpl['T_C'].values
        _yP = _p3_arcpl['P_kbar'].values
        _pT = _mT.predict(_Xt)
        _pP = _mP.predict(_Xp)
        _pred_df = _p3_arcpl.drop(columns=['_key'], errors='ignore').copy()
        _pred_df['T_pred'] = _pT
        _pred_df['P_pred'] = _pP
        _pred_df['family'] = _fam
        _pred_df.to_csv(RESULTS / f'nb04_arcpl_opx_liq_predictions_{_fam}.csv', index=False)
        # backward-compat alias
        _pred_df.to_csv(RESULTS / f'nb04b_arcpl_predictions_{_fam}.csv', index=False)
        _p3_pred_store[_fam] = _pred_df
        for _tgt, _y, _p in [('T_C', _yT, _pT), ('P_kbar', _yP, _pP)]:
            _m = _p3_metrics(_y, _p)
            _p3_metrics_rows.append({'family': _fam, 'target': _tgt, **_m})
            _r, _lo, _hi, _n = _p3_bootstrap_rmse(_y, _p)
            _p3_boot_rows.append({'family': _fam, 'target': _tgt,
                                   'rmse': _r, 'rmse_ci_lo': _lo, 'rmse_ci_hi': _hi, 'n': _n})
    except Exception as _e:
        print(f'Part 3 family {_fam} skipped ({_e})')

# Merged backward-compat file (both families in one CSV; disambiguate via `family`)
if _p3_pred_store:
    pd.concat(list(_p3_pred_store.values()), ignore_index=True).to_csv(
        RESULTS / 'nb04b_arcpl_predictions.csv', index=False)

_p3_metrics_df = pd.DataFrame(_p3_metrics_rows)
_p3_boot_df = pd.DataFrame(_p3_boot_rows)
_p3_metrics_df.to_csv(RESULTS / 'nb04_arcpl_opx_liq_metrics.csv', index=False)
_p3_boot_df.to_csv(RESULTS / 'nb04_arcpl_opx_liq_bootstrap.csv', index=False)
print(_p3_metrics_df.round(3).to_string(index=False))
print(_p3_boot_df.round(3).to_string(index=False))


# v8-fix: Part 3 scatter figure
import matplotlib.pyplot as _p3_plt
from src.plot_style import apply_style as _p3_apply_style

_p3_apply_style()
_p3_col_forest  = '#0072B2'
_p3_col_boosted = '#D55E00'

_p3_yT = _p3_arcpl['T_C'].values
_p3_yP = _p3_arcpl['P_kbar'].values

_p3_fig, _p3_axes = _p3_plt.subplots(2, 4, figsize=(16, 8))
for _col, (_fam, _color) in enumerate([('forest', _p3_col_forest),
                                        ('boosted', _p3_col_boosted)]):
    if _fam not in _p3_pred_store:
        continue
    _pd_f = _p3_pred_store[_fam].reset_index(drop=True)
    _tp = _pd_f['T_pred'].values
    _pp = _pd_f['P_pred'].values
    _tr = _tp - _p3_yT
    _pr = _pp - _p3_yP
    _rT = float(np.sqrt(np.nanmean(_tr ** 2)))
    _rP = float(np.sqrt(np.nanmean(_pr ** 2)))
    _base = _col * 2
    # [0, base]: pred vs obs T
    _a = _p3_axes[0, _base]
    _a.scatter(_p3_yT, _tp, s=12, alpha=0.6, color=_color, edgecolor='none')
    _lo, _hi = min(_p3_yT.min(), _tp.min()), max(_p3_yT.max(), _tp.max())
    _a.plot([_lo, _hi], [_lo, _hi], 'k--', lw=0.8, alpha=0.5)
    _a.set_xlabel('Observed T (C)'); _a.set_ylabel('Predicted T (C)')
    _a.set_title(f'{_fam.capitalize()} T (RMSE={_rT:.1f} C)')
    # [0, base+1]: pred vs obs P
    _a = _p3_axes[0, _base + 1]
    _a.scatter(_p3_yP, _pp, s=12, alpha=0.6, color=_color, edgecolor='none')
    _lo, _hi = min(_p3_yP.min(), _pp.min()), max(_p3_yP.max(), _pp.max())
    _a.plot([_lo, _hi], [_lo, _hi], 'k--', lw=0.8, alpha=0.5)
    _a.set_xlabel('Observed P (kbar)'); _a.set_ylabel('Predicted P (kbar)')
    _a.set_title(f'{_fam.capitalize()} P (RMSE={_rP:.2f} kbar)')
    # [1, base]: T residual hist
    _a = _p3_axes[1, _base]
    _a.hist(_tr[np.isfinite(_tr)], bins=30, color=_color, edgecolor='black',
            linewidth=0.3, alpha=0.8)
    _a.axvline(0, color='k', lw=0.8, alpha=0.5)
    _a.set_xlabel('T residual (pred - obs, C)'); _a.set_ylabel('count')
    _a.set_title(f'{_fam.capitalize()} T residuals')
    # [1, base+1]: P residual hist
    _a = _p3_axes[1, _base + 1]
    _a.hist(_pr[np.isfinite(_pr)], bins=30, color=_color, edgecolor='black',
            linewidth=0.3, alpha=0.8)
    _a.axvline(0, color='k', lw=0.8, alpha=0.5)
    _a.set_xlabel('P residual (pred - obs, kbar)'); _a.set_ylabel('count')
    _a.set_title(f'{_fam.capitalize()} P residuals')

_p3_fig.suptitle(f'ArcPL opx-liq external validation (n={len(_p3_arcpl)})',
                 fontsize=12, fontweight='bold', y=1.00)
_p3_plt.tight_layout()
for _ext in ('png', 'pdf'):
    _p3_fig.savefig(FIGURES / f'fig_nb04_arcpl_opx_liq_scatter.{_ext}',
                    dpi=300 if _ext == 'png' else None, bbox_inches='tight')
_p3_plt.show()
_p3_plt.close(_p3_fig)
print('Saved figures/fig_nb04_arcpl_opx_liq_scatter.{png,pdf}')


In [ ]:
# v8-fix: Restore nb04b cell-19 headline figure -- ML vs Putirka
# 3x2 panel comparison on ArcPL opx-liq scope (forest family).
# Reuses _p3_arcpl and _p3_pred_store from Part 3 above.
import matplotlib.pyplot as _pvm_plt
import numpy as _pvm_np
import pandas as _pvm_pd
import Thermobar as _pvm_tb
from src.plot_style import apply_style as _pvm_apply_style

_pvm_apply_style()

_pvm_df = _p3_arcpl.reset_index(drop=True).copy()
_pvm_yT = _pvm_df['T_C'].values
_pvm_yP = _pvm_df['P_kbar'].values
_pvm_n_full = len(_pvm_df)

# ---- ML predictions (forest family, headline canonical model) ------------
assert 'forest' in _p3_pred_store, 'forest predictions missing from _p3_pred_store'
_pvm_ml = _p3_pred_store['forest'].reset_index(drop=True)
_pvm_ml_T = _pvm_ml['T_pred'].values
_pvm_ml_P = _pvm_ml['P_pred'].values

# ---- Putirka 28a/29a iterative on the same rows (working recipe from
#      nb04 cell 32 encyclopedia; NOT the broken one from cell 21) -------
# v8-fix2: always build full Thermobar schema, defaulting missing cols to 0.
# Mirrors the working recipe from nb04 cell 6 (`g()` helper pattern).
def _pvm_g(col, n):
    if col in _pvm_df.columns:
        return _pvm_pd.to_numeric(_pvm_df[col], errors='coerce').fillna(0.0).values.astype(float)
    return _pvm_np.zeros(n, dtype=float)

_pvm_n = len(_pvm_df)
_pvm_opx_in = _pvm_pd.DataFrame({
    'SiO2_Opx':  _pvm_g('SiO2', _pvm_n),
    'TiO2_Opx':  _pvm_g('TiO2', _pvm_n),
    'Al2O3_Opx': _pvm_g('Al2O3', _pvm_n),
    'FeOt_Opx':  _pvm_g('FeO_total', _pvm_n),
    'MgO_Opx':   _pvm_g('MgO', _pvm_n),
    'CaO_Opx':   _pvm_g('CaO', _pvm_n),
    'MnO_Opx':   _pvm_g('MnO', _pvm_n),
    'Cr2O3_Opx': _pvm_g('Cr2O3', _pvm_n),
    'Na2O_Opx':  _pvm_g('Na2O', _pvm_n),
})
_pvm_liq_in = _pvm_pd.DataFrame({
    'SiO2_Liq':  _pvm_g('liq_SiO2', _pvm_n),
    'TiO2_Liq':  _pvm_g('liq_TiO2', _pvm_n),
    'Al2O3_Liq': _pvm_g('liq_Al2O3', _pvm_n),
    'FeOt_Liq':  _pvm_g('liq_FeO', _pvm_n),
    'MgO_Liq':   _pvm_g('liq_MgO', _pvm_n),
    'CaO_Liq':   _pvm_g('liq_CaO', _pvm_n),
    'Na2O_Liq':  _pvm_g('liq_Na2O', _pvm_n),
    'K2O_Liq':   _pvm_g('liq_K2O', _pvm_n),
    'MnO_Liq':   _pvm_g('liq_MnO', _pvm_n),
    'Cr2O3_Liq': _pvm_g('liq_Cr2O3', _pvm_n),
    'P2O5_Liq':  _pvm_g('P2O5_Liq', _pvm_n),
    'H2O_Liq':   _pvm_g('H2O_Liq', _pvm_n),
})
print(f'Put-schema: opx_in {_pvm_opx_in.shape}, liq_in {_pvm_liq_in.shape}')

# v8-fix: Fe3Fet_Liq REQUIRED by eq28a/29a -- use 0.0 (reduced) per spec.
_pvm_liq_in['Fe3Fet_Liq'] = 0.0

try:
    _pvm_out = _pvm_tb.calculate_opx_liq_press_temp(
        opx_comps=_pvm_opx_in, liq_comps=_pvm_liq_in,
        equationP='P_Put2008_eq29a', equationT='T_Put2008_eq28a',
    )
    _pvm_put_T = _pvm_np.asarray(_pvm_out['T_K_calc']) - 273.15
    _pvm_put_P = _pvm_np.asarray(_pvm_out['P_kbar_calc'])
except Exception as _pvm_e:
    print(f'Putirka eq28a/29a run failed: {_pvm_e}')
    _pvm_put_T = _pvm_np.full(_pvm_n_full, _pvm_np.nan)
    _pvm_put_P = _pvm_np.full(_pvm_n_full, _pvm_np.nan)

# ---- Fair subset: rows where Putirka returns finite T AND P -------------
_pvm_ok = _pvm_np.isfinite(_pvm_put_T) & _pvm_np.isfinite(_pvm_put_P)
_pvm_n_fair = int(_pvm_ok.sum())
_pvm_fail_rate = 100.0 * (1.0 - _pvm_n_fair / _pvm_n_full)
print(f'Putirka convergence: fair n={_pvm_n_fair}/{_pvm_n_full} '
      f'({100.0 * _pvm_n_fair / _pvm_n_full:.1f}% coverage, '
      f'failure rate {_pvm_fail_rate:.1f}%)')


def _pvm_rmse(a, b):
    a = _pvm_np.asarray(a, dtype=float); b = _pvm_np.asarray(b, dtype=float)
    m = _pvm_np.isfinite(a) & _pvm_np.isfinite(b)
    return float(_pvm_np.sqrt(_pvm_np.mean((a[m] - b[m]) ** 2))) if m.any() else _pvm_np.nan


# ---- Figure: 3 rows x 2 cols (T left, P right) --------------------------
_pvm_fig, _pvm_ax = _pvm_plt.subplots(3, 2, figsize=(11, 14),
                                      constrained_layout=False)
_pvm_COL_ML = '#0072B2'   # blue
_pvm_COL_PU = '#D55E00'   # orange

# T axis range: pool observed + all predictions
_pvm_T_pool = _pvm_np.concatenate([_pvm_yT, _pvm_ml_T, _pvm_put_T[_pvm_ok]])
_pvm_T_pool = _pvm_T_pool[_pvm_np.isfinite(_pvm_T_pool)]
_pvm_T_lo, _pvm_T_hi = float(_pvm_T_pool.min()) - 30, float(_pvm_T_pool.max()) + 30

_pvm_P_pool = _pvm_np.concatenate([_pvm_yP, _pvm_ml_P, _pvm_put_P[_pvm_ok]])
_pvm_P_pool = _pvm_P_pool[_pvm_np.isfinite(_pvm_P_pool)]
_pvm_P_lo, _pvm_P_hi = float(_pvm_P_pool.min()) - 1, float(_pvm_P_pool.max()) + 1


def _pvm_panel(ax, x, y, color, title, unit, lo, hi, extra_text=None):
    m = _pvm_np.isfinite(x) & _pvm_np.isfinite(y)
    ax.scatter(x[m], y[m], s=16, alpha=0.6, color=color, edgecolor='none')
    ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, alpha=0.5)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel(f'Observed ({unit})')
    ax.set_ylabel(f'Predicted ({unit})')
    ax.set_title(title, fontsize=10.5)
    ax.grid(True, alpha=0.3)
    if extra_text:
        ax.text(0.03, 0.97, extra_text, transform=ax.transAxes,
                va='top', ha='left', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white',
                          edgecolor='0.6', alpha=0.85))


# Row 1: ML on ALL samples (n=197, 100% coverage) -------------------------
_pvm_rmse_T_all = _pvm_rmse(_pvm_ml_T, _pvm_yT)
_pvm_rmse_P_all = _pvm_rmse(_pvm_ml_P, _pvm_yP)
_pvm_panel(_pvm_ax[0, 0], _pvm_yT, _pvm_ml_T, _pvm_COL_ML,
           f'ML forest T - all ArcPL (n={_pvm_n_full}, 100% coverage)\n'
           f'RMSE = {_pvm_rmse_T_all:.1f} C',
           'C', _pvm_T_lo, _pvm_T_hi)
_pvm_panel(_pvm_ax[0, 1], _pvm_yP, _pvm_ml_P, _pvm_COL_ML,
           f'ML forest P - all ArcPL (n={_pvm_n_full}, 100% coverage)\n'
           f'RMSE = {_pvm_rmse_P_all:.2f} kbar',
           'kbar', _pvm_P_lo, _pvm_P_hi)

# Row 2: ML on fair subset (where Putirka converges) ----------------------
_pvm_rmse_T_fair = _pvm_rmse(_pvm_ml_T[_pvm_ok], _pvm_yT[_pvm_ok])
_pvm_rmse_P_fair = _pvm_rmse(_pvm_ml_P[_pvm_ok], _pvm_yP[_pvm_ok])
_pvm_panel(_pvm_ax[1, 0], _pvm_yT[_pvm_ok], _pvm_ml_T[_pvm_ok], _pvm_COL_ML,
           f'ML forest T - fair subset (n={_pvm_n_fair})\n'
           f'RMSE = {_pvm_rmse_T_fair:.1f} C',
           'C', _pvm_T_lo, _pvm_T_hi)
_pvm_panel(_pvm_ax[1, 1], _pvm_yP[_pvm_ok], _pvm_ml_P[_pvm_ok], _pvm_COL_ML,
           f'ML forest P - fair subset (n={_pvm_n_fair})\n'
           f'RMSE = {_pvm_rmse_P_fair:.2f} kbar',
           'kbar', _pvm_P_lo, _pvm_P_hi)

# Row 3: Putirka 28a/29a on fair subset -----------------------------------
_pvm_rmse_T_put = _pvm_rmse(_pvm_put_T[_pvm_ok], _pvm_yT[_pvm_ok])
_pvm_rmse_P_put = _pvm_rmse(_pvm_put_P[_pvm_ok], _pvm_yP[_pvm_ok])
_pvm_fail_text = f'Putirka failure rate on full set: {_pvm_fail_rate:.1f}%'
_pvm_panel(_pvm_ax[2, 0], _pvm_yT[_pvm_ok], _pvm_put_T[_pvm_ok], _pvm_COL_PU,
           f'Putirka 2008 eq28a T - fair subset (n={_pvm_n_fair})\n'
           f'RMSE = {_pvm_rmse_T_put:.1f} C',
           'C', _pvm_T_lo, _pvm_T_hi, extra_text=_pvm_fail_text)
_pvm_panel(_pvm_ax[2, 1], _pvm_yP[_pvm_ok], _pvm_put_P[_pvm_ok], _pvm_COL_PU,
           f'Putirka 2008 eq29a P - fair subset (n={_pvm_n_fair})\n'
           f'RMSE = {_pvm_rmse_P_put:.2f} kbar',
           'kbar', _pvm_P_lo, _pvm_P_hi, extra_text=_pvm_fail_text)

_pvm_fig.suptitle(
    'ArcPL opx-liq: ML forest vs Putirka 2008 (restored nb04b headline)',
    fontsize=13, fontweight='bold', y=0.995)
_pvm_fig.tight_layout(rect=[0, 0.06, 1, 0.975])

_pvm_caption = (
    f'Top: ML predictions on all n={_pvm_n_full} ArcPL opx-bearing experiments '
    f'(100% coverage). Middle: ML predictions on the n={_pvm_n_fair} subset '
    f'where Putirka 2008 eq 28a/29a converges. Bottom: Putirka predictions '
    f'on the same fair subset. Putirka fails to predict on '
    f'{_pvm_fail_rate:.1f}% of the full dataset due to equilibrium filter '
    f'rejections.'
)
_pvm_plt.figtext(0.5, 0.02, _pvm_caption, ha='center', va='bottom',
                 fontsize=9, wrap=True)

for _ext in ('png', 'pdf'):
    _pvm_fig.savefig(FIGURES / f'fig_nb04_putirka_vs_ml_arcpl.{_ext}',
                     dpi=300 if _ext == 'png' else None, bbox_inches='tight')
_pvm_plt.show()
_pvm_plt.close(_pvm_fig)

print(f'Saved figures/fig_nb04_putirka_vs_ml_arcpl.{{png,pdf}}')
print(f'  Row 1 (ML all):       T RMSE={_pvm_rmse_T_all:.1f} C  P RMSE={_pvm_rmse_P_all:.2f} kbar')
print(f'  Row 2 (ML fair):      T RMSE={_pvm_rmse_T_fair:.1f} C  P RMSE={_pvm_rmse_P_fair:.2f} kbar')
print(f'  Row 3 (Putirka fair): T RMSE={_pvm_rmse_T_put:.1f} C  P RMSE={_pvm_rmse_P_put:.2f} kbar')
# Report whichever method wins P vs T on the fair subset; no hard assert.
print(f'Fair-subset verdict: ML wins P if {_pvm_rmse_P_fair:.2f} < {_pvm_rmse_P_put:.2f}; '
      f'Putirka wins T if {_pvm_rmse_T_put:.1f} < {_pvm_rmse_T_fair:.1f}.')


<!-- v7-fix-section-header -->
## Part 4: H2O reporting sensitivity analysis

**Purpose.** Check whether our ML's ArcPL performance differs between
samples with directly measured H2O (FTIR/SIMS/Raman/Sol) versus those
with VBD/mass-balance H2O. Relevant because many ArcPL experiments do
not have measured H2O.

**Data inputs.** The cleaned ArcPL opx-liq frame built in Part 3 (n~=204).
Partition by the `is_vbd` flag derived from `H2O_Liq_Method`.

**Methods evaluated.** Canonical opx-liq forest and boosted, per-subset.

**Analysis performed.** For each subset (measured-H2O ~=134, VBD ~=70)
compute RMSE, MAE, R^2 with 95% bootstrap CI (B=2000). Run Mann-Whitney U
on absolute residuals between subsets for T and P separately. Produce a
4-panel figure: T pred-vs-obs by subset, P pred-vs-obs by subset, and
residual histograms for T and P.

**How to interpret.** Robust performance across H2O methods supports
general-use applicability. A large gap (especially measured << VBD) would
suggest H2O quality matters and would motivate a user caveat. VBD samples
often have systematically lower P, which confounds P comparisons; any
apparent VBD advantage is likely this artifact, not a real model property.

**Outputs.**
- `results/nb04_arcpl_h2o_stratified_metrics.csv`
- `results/nb04_arcpl_h2o_mannwhitney.csv`
- `figures/fig_nb04_h2o_sensitivity.{png,pdf}`

**Downstream use.** NB09 may cite the p-values in a supplementary discussion
paragraph. Terminal otherwise — not consumed by other notebooks.


In [ ]:
# v7-fix: Part 4 H2O reporting sensitivity analysis
# Uses `_p3_arcpl` + `_p3_pred_store` produced in Part 3.
from scipy.stats import mannwhitneyu as _p4_mwu
import matplotlib.pyplot as _p4_plt

_p4_rows = []
_p4_mw_rows = []

if not _p3_pred_store:
    print('Part 4 skipped (no Part 3 predictions).')
else:
    _p4_df_base = _p3_arcpl.reset_index(drop=True)
    _p4_meas_mask = ~_p4_df_base['is_vbd'].values.astype(bool)
    _p4_vbd_mask  =  _p4_df_base['is_vbd'].values.astype(bool)
    print(f'Part 4 split: measured n={int(_p4_meas_mask.sum())} '
          f'VBD n={int(_p4_vbd_mask.sum())}')

    for _fam, _pred_df in _p3_pred_store.items():
        _pred_df = _pred_df.reset_index(drop=True)
        _yT = _pred_df['T_C'].values
        _yP = _pred_df['P_kbar'].values
        _pT = _pred_df['T_pred'].values
        _pP = _pred_df['P_pred'].values
        for _sub_name, _mask in [('measured', _p4_meas_mask),
                                  ('vbd',      _p4_vbd_mask)]:
            if _mask.sum() < 5:
                continue
            for _tgt, _y, _p in [('T_C', _yT, _pT), ('P_kbar', _yP, _pP)]:
                _m = _p3_metrics(_y[_mask], _p[_mask])
                _r, _lo, _hi, _n = _p3_bootstrap_rmse(_y[_mask], _p[_mask])
                _p4_rows.append({'family': _fam, 'subset': _sub_name,
                                  'target': _tgt, **_m,
                                  'rmse_ci_lo': _lo, 'rmse_ci_hi': _hi})
        # Mann-Whitney on |residuals|
        for _tgt, _resid in [('T_C', _pT - _yT), ('P_kbar', _pP - _yP)]:
            _a = np.abs(_resid[_p4_meas_mask])
            _b = np.abs(_resid[_p4_vbd_mask])
            _a = _a[np.isfinite(_a)]; _b = _b[np.isfinite(_b)]
            if len(_a) >= 5 and len(_b) >= 5:
                _s, _pp = _p4_mwu(_a, _b, alternative='two-sided')
                _p4_mw_rows.append({'family': _fam, 'target': _tgt,
                                     'n_measured': int(len(_a)),
                                     'n_vbd': int(len(_b)),
                                     'U_stat': float(_s),
                                     'p_value': float(_pp),
                                     'median_abs_resid_measured': float(np.median(_a)),
                                     'median_abs_resid_vbd':      float(np.median(_b))})

    _p4_metrics_df = pd.DataFrame(_p4_rows)
    _p4_mw_df = pd.DataFrame(_p4_mw_rows)
    _p4_metrics_df.to_csv(RESULTS / 'nb04_arcpl_h2o_stratified_metrics.csv', index=False)
    _p4_mw_df.to_csv(RESULTS / 'nb04_arcpl_h2o_mannwhitney.csv', index=False)
    print('\nStratified metrics:')
    print(_p4_metrics_df.round(3).to_string(index=False))
    print('\nMann-Whitney:')
    print(_p4_mw_df.round(4).to_string(index=False))

    # 4-panel figure using the forest family if available, else boosted.
    _p4_fam = 'forest' if 'forest' in _p3_pred_store else list(_p3_pred_store)[0]
    _pd = _p3_pred_store[_p4_fam].reset_index(drop=True)
    _yT = _pd['T_C'].values; _yP = _pd['P_kbar'].values
    _pT = _pd['T_pred'].values; _pP = _pd['P_pred'].values
    _fig, _axes = _p4_plt.subplots(2, 2, figsize=(11, 9))
    _col_meas = '#0072B2'; _col_vbd = '#D55E00'

    def _p4_scatter(ax, y, p, mask_m, mask_v, unit):
        ax.scatter(y[mask_m], p[mask_m], s=18, alpha=0.6, c=_col_meas,
                   label=f'measured n={int(mask_m.sum())}', edgecolor='k', lw=0.2)
        ax.scatter(y[mask_v], p[mask_v], s=18, alpha=0.6, c=_col_vbd,
                   label=f'VBD n={int(mask_v.sum())}', edgecolor='k', lw=0.2)
        _lim = [np.nanmin([y.min(), p.min()]), np.nanmax([y.max(), p.max()])]
        ax.plot(_lim, _lim, 'k--', lw=0.8)
        _rm = float(np.sqrt(np.nanmean((y[mask_m] - p[mask_m]) ** 2))) if mask_m.sum() else float('nan')
        _rv = float(np.sqrt(np.nanmean((y[mask_v] - p[mask_v]) ** 2))) if mask_v.sum() else float('nan')
        ax.set_xlabel(f'Observed ({unit})'); ax.set_ylabel(f'Predicted ({unit})')
        ax.set_title(f'RMSE meas={_rm:.2f} / VBD={_rv:.2f} ({unit})')
        ax.legend(loc='best', fontsize=8); ax.grid(True, alpha=0.3)

    _p4_scatter(_axes[0, 0], _yT, _pT, _p4_meas_mask, _p4_vbd_mask, 'C')
    _p4_scatter(_axes[0, 1], _yP, _pP, _p4_meas_mask, _p4_vbd_mask, 'kbar')
    _axes[0, 0].set_title('T: ' + _axes[0, 0].get_title())
    _axes[0, 1].set_title('P: ' + _axes[0, 1].get_title())
    for _ax, _y, _p, _unit in [
        (_axes[1, 0], _yT, _pT, 'C'),
        (_axes[1, 1], _yP, _pP, 'kbar'),
    ]:
        _resid = _p - _y
        _ax.hist(_resid[_p4_meas_mask], bins=25, alpha=0.6, color=_col_meas,
                 label='measured', density=True)
        _ax.hist(_resid[_p4_vbd_mask], bins=25, alpha=0.6, color=_col_vbd,
                 label='VBD', density=True)
        _ax.axvline(0, color='k', lw=0.8)
        _ax.set_xlabel(f'Residual ({_unit})')
        _ax.set_ylabel('density')
        _ax.legend(loc='best', fontsize=8); _ax.grid(True, alpha=0.3)
    _p4_plt.suptitle(f'H2O sensitivity - {_p4_fam} family', fontsize=12)
    _p4_plt.tight_layout()
    for _ext in ('png', 'pdf'):
        _p4_plt.savefig(FIGURES / f'fig_nb04_h2o_sensitivity.{_ext}',
                         dpi=300, bbox_inches='tight')
    _p4_plt.close(_fig)


<!-- v7-fix-section-header -->
## Part 5: Opx-only method comparison, dual-scope (headline figure)

**Purpose.** The paper's headline deployment figure. Answers two questions
at once: (i) across all ArcPL opx-bearing samples, how do our ML methods
compare to Putirka opx methods in deployment terms (coverage + RMSE on
what each method can return)? (ii) on the narrow subset where Putirka's
equilibrium filters permit a prediction ("fair scope"), does our ML beat
Putirka head-to-head?

**Data inputs.** The Part 3 cleaned ArcPL opx-liq frame (n~=204, "all" scope).
The "fair" scope is defined dynamically as rows where Putirka 28a/29a iterative
returns finite T *and* P — typically n~=28.

**Methods evaluated.** 8 total:
1. Ours opx-liq forest (canonical RF on alr for T, RF on raw for P)
2. Ours opx-liq boosted (canonical XGB on raw for T and P)
3. Ours opx-only forest (canonical RF on pwlr for T and P)
4. Ours opx-only boosted (canonical XGB on pwlr for T and P)
5. Putirka 2008 opx-liq (eq 28a + eq 29a iterative)
6. Putirka 2008 opx-only (eq 28b_opx_sat + eq 29c iterative, Cr2O3_Opx > 0)
7. Putirka 2008 opx-liq [true P] (eq 28a with true P input -> ceiling)
8. Putirka 2008 opx-only [true T] (eq 28b_opx_sat with true T input -> ceiling)

**Analysis performed.** Compute RMSE + 95% bootstrap CI (B=2000), signed bias,
and coverage (n_predicted / n_scope * 100) for every (method, scope) pair.
Produce a two-panel RMSE bar chart (T on the left, P on the right, 8 methods
each, sorted by T RMSE in panel A). Putirka bars are color-coded sky blue and
hatched; ML bars use family colors (forest blue, boosted vermillion). Two
side-by-side bars per method distinguish "all" scope (alpha=0.6) from "fair"
scope (alpha=1.0). A separate coverage panel shows n_predicted/n_scope.

**How to interpret.** On the "all" scope (n~=204), coverage exposes Putirka's
real-world limitation: iterative methods drop to ~13.7% due to equilibrium
filter failures, while our ML returns predictions on 100% of opx-bearing arc
samples. On the "fair" scope (n~=28), RMSE is apples-to-apples. The expected
outcome is that our ML has lower RMSE on both T and P than Putirka iterative,
by a factor near 2.6x on T (~=46 C vs ~=119 C). The "true P" and "true T"
Putirka variants are CEILING comparisons — they use the correct answer as
input, representing the best Putirka could ever do; real users do not have
true P/T when predicting. If our ML beats Putirka even when Putirka is given
the true answer, the deployment argument is complete: competitive with the
ceiling AND always-on.

**Outputs.**
- `results/nb04_opx_only_comparison_all.csv`  (8 methods, n_all scope)
- `results/nb04_opx_only_comparison_fair.csv` (8 methods, n_fair scope)
- `figures/fig_nb04_opx_only_comparison.{png,pdf}`

**Downstream use.** NB09 pulls the headline "ML ~=46 C vs Putirka ~=119 C"
number from the fair-scope CSV for the paper abstract.


In [ ]:
# v8-fix: Putirka opx corrected API
# v7-fix: Part 5 opx-only 8-method dual-scope comparison
import matplotlib.pyplot as _p5_plt
from src.features import lepr_to_training_features as _p5_shim
import joblib as _p5_joblib

if not _p3_pred_store:
    print('Part 5 skipped (no Part 3 predictions).')
else:
    _p5_df = _p3_arcpl.reset_index(drop=True).copy()
    # Part 5 treats _p5_df as a LEPR-suffix frame; Part 3 already renamed
    # to flat training-schema (SiO2, liq_SiO2, etc.), so recover the LEPR
    # suffixed columns Putirka expects from _p3_lepr joined on Experiment.
    _p5_raw = _p3_lepr.copy()
    if 'Experiment' not in _p5_raw.columns and 'Experiment_x' in _p5_raw.columns:
        _p5_raw = _p5_raw.rename(columns={'Experiment_x': 'Experiment'})
    _p5_merged = _p5_df.merge(
        _p5_raw[['Experiment'] + [c for c in _p5_raw.columns
                                   if c.endswith('_Opx') or c.endswith('_Liq')]],
        on='Experiment', how='left', suffixes=('', '_lepr'))
    _yT_all = _p5_df['T_C'].values; _yP_all = _p5_df['P_kbar'].values

    _p5_preds_all = {}

    # Ours predictions: reuse Part 3 for opx-liq; recompute opx-only locally.
    for _fam in ['forest', 'boosted']:
        if _fam in _p3_pred_store:
            _pd = _p3_pred_store[_fam].reset_index(drop=True)
            _p5_preds_all[f'Ours opx-liq {_fam}'] = (_pd['T_pred'].values,
                                                      _pd['P_pred'].values)
        try:
            _sT = _p3_canon_spec('T_C',    'opx_only', _fam, RESULTS)
            _sP = _p3_canon_spec('P_kbar', 'opx_only', _fam, RESULTS)
            _mT = _p5_joblib.load(_p3_canon_path('T_C',    'opx_only', _fam, MODELS, RESULTS))
            _mP = _p5_joblib.load(_p3_canon_path('P_kbar', 'opx_only', _fam, MODELS, RESULTS))
            _Xt, _ = _p3_build_feat(_p5_df, _sT['feature_set'], use_liq=False)
            _Xp, _ = _p3_build_feat(_p5_df, _sP['feature_set'], use_liq=False)
            _p5_preds_all[f'Ours opx-only {_fam}'] = (_mT.predict(_Xt), _mP.predict(_Xp))
        except Exception as _e:
            print(f'Part 5 Ours opx-only {_fam} skipped ({_e})')

    # Putirka: run four variants off _p5_merged LEPR-suffixed columns.
    try:
        import Thermobar as _p5_pt
        _opx_in = pd.DataFrame({
            k: pd.to_numeric(_p5_merged.get(k, np.nan), errors='coerce')
            for k in ['SiO2_Opx','TiO2_Opx','Al2O3_Opx','Cr2O3_Opx','FeOt_Opx',
                      'MnO_Opx','MgO_Opx','CaO_Opx','Na2O_Opx','K2O_Opx']
            if k in _p5_merged.columns})
        _liq_in = pd.DataFrame({
            k: pd.to_numeric(_p5_merged.get(k, np.nan), errors='coerce')
            for k in ['SiO2_Liq','TiO2_Liq','Al2O3_Liq','FeOt_Liq','MnO_Liq',
                      'MgO_Liq','CaO_Liq','Na2O_Liq','K2O_Liq','Cr2O3_Liq','P2O5_Liq']
            if k in _p5_merged.columns})
        if 'H2O_Liq' in _p5_merged.columns:
            _liq_in['H2O_Liq'] = pd.to_numeric(_p5_merged['H2O_Liq'], errors='coerce').fillna(0.0)
            # v8-fix: Putirka opx-liq eq28a/29a require Fe3Fet_Liq
            _liq_in['Fe3Fet_Liq'] = 0.0
        # (5) opx-liq iterative
        try:
            _it = _p5_pt.calculate_opx_liq_press_temp(
                opx_comps=_opx_in, liq_comps=_liq_in,
                equationT='T_Put2008_eq28a', equationP='P_Put2008_eq29a')
            _T = (_it['T_K_calc'].values - 273.15 if 'T_K_calc' in _it.columns
                  else _it.iloc[:, 0].values - 273.15)
            _P = (_it['P_kbar_calc'].values if 'P_kbar_calc' in _it.columns
                  else _it.iloc[:, 1].values)
            _p5_preds_all['Putirka 2008 opx-liq'] = (_T, _P)
        except Exception as _e:
            print(f'Part 5 Putirka opx-liq iterative skipped ({_e})')
        # (7) opx-liq ceiling with true P, true T
        try:
            _Tc = _p5_pt.calculate_opx_liq_temp(
                opx_comps=_opx_in, liq_comps=_liq_in,
                equationT='T_Put2008_eq28a', P=_yP_all)
            _Pc = _p5_pt.calculate_opx_liq_press(
                opx_comps=_opx_in, liq_comps=_liq_in,
                equationP='P_Put2008_eq29a', T=_yT_all + 273.15)
            _Tc_arr = (_Tc.values - 273.15 if hasattr(_Tc, 'values')
                       else np.asarray(_Tc) - 273.15)
            _Pc_arr = (_Pc.values if hasattr(_Pc, 'values') else np.asarray(_Pc))
            _p5_preds_all['Putirka 2008 opx-liq [true P]'] = (_Tc_arr, _Pc_arr)
        except Exception as _e:
            print(f'Part 5 Putirka opx-liq [true P] skipped ({_e})')
        # (6) opx-only iterative with Cr2O3 > 0 filter
        try:
            _mask_cr = ((_opx_in.get('Cr2O3_Opx', pd.Series(np.zeros(len(_p5_merged))))
                         .fillna(0.0) > 1e-4).values
                        if 'Cr2O3_Opx' in _opx_in.columns
                        else np.zeros(len(_p5_merged), dtype=bool))
            if _mask_cr.sum() >= 3:
                _opx_cr = _opx_in[_mask_cr].reset_index(drop=True)
                # v8-fix: no calculate_opx_only_press_temp in Thermobar.
                _liq_cr = _liq_in[_mask_cr].reset_index(drop=True)
                _T_init = _p5_pt.calculate_opx_liq_temp(
                    opx_comps=_opx_cr, liq_comps=_liq_cr,
                    equationT='T_Put2008_eq28a', P=10.0)
                _T_init_K = (_T_init['T_K_calc'].values
                             if hasattr(_T_init, 'columns') and 'T_K_calc' in _T_init.columns
                             else (_T_init.values if hasattr(_T_init, 'values')
                                   else np.asarray(_T_init)))
                if np.nanmedian(_T_init_K) < 400:
                    _T_init_K = _T_init_K + 273.15
                _P_step = _p5_pt.calculate_opx_only_press(
                    opx_comps=_opx_cr, equationP='P_Put2008_eq29c', T=_T_init_K)
                _P_arr = (_P_step['P_kbar_calc'].values
                          if hasattr(_P_step, 'columns') and 'P_kbar_calc' in _P_step.columns
                          else (_P_step.values if hasattr(_P_step, 'values')
                                else np.asarray(_P_step)))
                _T_step = _p5_pt.calculate_opx_liq_temp(
                    opx_comps=_opx_cr, liq_comps=_liq_cr,
                    equationT='T_Put2008_eq28b_opx_sat', P=_P_arr)
                _T_K = (_T_step['T_K_calc'].values
                        if hasattr(_T_step, 'columns') and 'T_K_calc' in _T_step.columns
                        else (_T_step.values if hasattr(_T_step, 'values')
                              else np.asarray(_T_step)))
                _T_arr = _T_K - 273.15 if np.nanmedian(_T_K) > 400 else _T_K
                _To = np.full(len(_p5_merged), np.nan)
                _Po = np.full(len(_p5_merged), np.nan)
                _To[_mask_cr] = _T_arr
                _Po[_mask_cr] = _P_arr
                _p5_preds_all['Putirka 2008 opx-only'] = (_To, _Po)
                # (8) opx-only ceiling with true T, true P
                try:
                    _Pc_o = _p5_pt.calculate_opx_only_press(
                        opx_comps=_opx_cr, equationP='P_Put2008_eq29c',
                        T=(_yT_all + 273.15)[_mask_cr])
                    _Tc_o = _p5_pt.calculate_opx_liq_temp(
                        opx_comps=_opx_cr, liq_comps=_liq_cr,
                        equationT='T_Put2008_eq28b_opx_sat',
                        P=_yP_all[_mask_cr])
                    _Po2 = np.full(len(_p5_merged), np.nan)
                    _To2 = np.full(len(_p5_merged), np.nan)
                    _Po2[_mask_cr] = (_Pc_o['P_kbar_calc'].values
                                      if hasattr(_Pc_o, 'columns') and 'P_kbar_calc' in _Pc_o.columns
                                      else (_Pc_o.values if hasattr(_Pc_o, 'values')
                                            else np.asarray(_Pc_o)))
                    _Tck2 = (_Tc_o['T_K_calc'].values
                             if hasattr(_Tc_o, 'columns') and 'T_K_calc' in _Tc_o.columns
                             else (_Tc_o.values if hasattr(_Tc_o, 'values')
                                   else np.asarray(_Tc_o)))
                    _To2[_mask_cr] = _Tck2 - 273.15 if np.nanmedian(_Tck2) > 400 else _Tck2
                    _p5_preds_all['Putirka 2008 opx-only [true T]'] = (_To2, _Po2)
                except Exception as _e:
                    print(f'Part 5 Putirka opx-only [true T] skipped ({_e})')
        except Exception as _e:
            print(f'Part 5 Putirka opx-only iterative skipped ({_e})')
    except Exception as _e:
        print(f'Part 5 Putirka block skipped ({_e})')

    # Fair-scope mask: rows where Putirka opx-liq iterative returned finite T and P.
    _pit = _p5_preds_all.get('Putirka 2008 opx-liq', (None, None))
    if _pit[0] is not None and _pit[1] is not None:
        _fair_mask = np.isfinite(_pit[0]) & np.isfinite(_pit[1])
    else:
        _fair_mask = np.zeros(len(_p5_df), dtype=bool)
    print(f'Part 5 scopes: all n={len(_p5_df)}  fair n={int(_fair_mask.sum())}')

    def _p5_row(name, y, yhat, scope_name, scope_mask):
        yhat = np.asarray(yhat, dtype=float)
        y = np.asarray(y, dtype=float)
        mask = scope_mask & np.isfinite(yhat) & np.isfinite(y)
        n = int(mask.sum())
        if n < 3:
            return dict(method=name, scope=scope_name, n=n, rmse=np.nan,
                        rmse_ci_lo=np.nan, rmse_ci_hi=np.nan,
                        coverage_pct=100.0 * n / max(int(scope_mask.sum()), 1),
                        signed_bias=np.nan)
        _rmse, _lo, _hi, _ = _p3_bootstrap_rmse(y[mask], yhat[mask])
        bias = float(np.mean(yhat[mask] - y[mask]))
        return dict(method=name, scope=scope_name, n=n, rmse=_rmse,
                    rmse_ci_lo=_lo, rmse_ci_hi=_hi,
                    coverage_pct=100.0 * n / max(int(scope_mask.sum()), 1),
                    signed_bias=bias)

    _all_mask = np.ones(len(_p5_df), dtype=bool)
    _rows_all_T, _rows_all_P, _rows_fair_T, _rows_fair_P = [], [], [], []
    for _name, (_pT_m, _pP_m) in _p5_preds_all.items():
        _rows_all_T.append(_p5_row(_name, _yT_all, _pT_m, 'all',  _all_mask))
        _rows_all_P.append(_p5_row(_name, _yP_all, _pP_m, 'all',  _all_mask))
        _rows_fair_T.append(_p5_row(_name, _yT_all, _pT_m, 'fair', _fair_mask))
        _rows_fair_P.append(_p5_row(_name, _yP_all, _pP_m, 'fair', _fair_mask))

    _df_all = pd.DataFrame([{**tr, 'target': 'T_C'} for tr in _rows_all_T] +
                           [{**pr, 'target': 'P_kbar'} for pr in _rows_all_P])
    _df_fair = pd.DataFrame([{**tr, 'target': 'T_C'} for tr in _rows_fair_T] +
                            [{**pr, 'target': 'P_kbar'} for pr in _rows_fair_P])
    _df_all.to_csv(RESULTS / 'nb04_opx_only_comparison_all.csv', index=False)
    _df_fair.to_csv(RESULTS / 'nb04_opx_only_comparison_fair.csv', index=False)
    print('\nPart 5 (all scope):')
    print(_df_all.round(3).to_string(index=False))
    print('\nPart 5 (fair scope):')
    print(_df_fair.round(3).to_string(index=False))

    # Two-panel bar chart + coverage side panel.
    from config import FAMILY_COLORS as _p5_FC
    _SKY = '#56B4E9'
    def _p5_color(name):
        if name.startswith('Ours') and 'forest' in name:
            return _p5_FC['forest']
        if name.startswith('Ours') and 'boosted' in name:
            return _p5_FC['boosted']
        return _SKY

    _methods = list(_p5_preds_all.keys())
    _fig, _axes = _p5_plt.subplots(1, 3, figsize=(16, 6),
                                     gridspec_kw={'width_ratios': [4, 4, 2]})

    def _panel(ax, rows_all, rows_fair, title, unit):
        methods_sorted = sorted(rows_all,
                                 key=lambda r: (r['rmse'] if np.isfinite(r['rmse']) else 1e9))
        order = [r['method'] for r in methods_sorted]
        y = np.arange(len(order))
        rmse_all = np.array([next(r['rmse'] for r in rows_all if r['method'] == m) for m in order])
        lo_all   = np.array([next(r['rmse_ci_lo'] for r in rows_all if r['method'] == m) for m in order])
        hi_all   = np.array([next(r['rmse_ci_hi'] for r in rows_all if r['method'] == m) for m in order])
        rmse_fair = np.array([next(r['rmse'] for r in rows_fair if r['method'] == m) for m in order])
        lo_f   = np.array([next(r['rmse_ci_lo'] for r in rows_fair if r['method'] == m) for m in order])
        hi_f   = np.array([next(r['rmse_ci_hi'] for r in rows_fair if r['method'] == m) for m in order])
        colors = [_p5_color(m) for m in order]
        hatches = ['///' if 'Putirka' in m else '' for m in order]
        # all scope: alpha 0.6
        for i, m in enumerate(order):
            ax.barh(y[i] - 0.2, rmse_all[i], height=0.4,
                    xerr=[[rmse_all[i]-lo_all[i] if np.isfinite(lo_all[i]) else 0],
                          [hi_all[i]-rmse_all[i] if np.isfinite(hi_all[i]) else 0]],
                    color=colors[i], alpha=0.5, edgecolor='k', lw=0.5,
                    hatch=hatches[i], error_kw=dict(ecolor='gray', lw=0.8))
            ax.barh(y[i] + 0.2, rmse_fair[i], height=0.4,
                    xerr=[[rmse_fair[i]-lo_f[i] if np.isfinite(lo_f[i]) else 0],
                          [hi_f[i]-rmse_fair[i] if np.isfinite(hi_f[i]) else 0]],
                    color=colors[i], alpha=1.0, edgecolor='k', lw=0.5,
                    hatch=hatches[i], error_kw=dict(ecolor='k', lw=0.8))
            if np.isfinite(rmse_all[i]):
                ax.text(rmse_all[i], y[i] - 0.2, f' {rmse_all[i]:.1f}',
                        va='center', fontsize=7, alpha=0.7)
            if np.isfinite(rmse_fair[i]):
                ax.text(rmse_fair[i], y[i] + 0.2, f' {rmse_fair[i]:.1f}',
                        va='center', fontsize=7)
        ax.set_yticks(y); ax.set_yticklabels(order, fontsize=8)
        ax.set_xlabel(f'RMSE ({unit})'); ax.set_title(title)
        ax.grid(True, alpha=0.3, axis='x')

    _panel(_axes[0], _rows_all_T, _rows_fair_T, 'T (sorted by all-scope T RMSE)', 'C')
    _panel(_axes[1], _rows_all_P, _rows_fair_P, 'P (sorted by all-scope P RMSE)', 'kbar')

    # Coverage panel: one group per method, two bars (all / fair)
    _order_cov = [r['method'] for r in sorted(_rows_all_T,
                                               key=lambda r: (r['rmse'] if np.isfinite(r['rmse']) else 1e9))]
    _cov_all = [next(r['coverage_pct'] for r in _rows_all_T if r['method'] == m) for m in _order_cov]
    _cov_fair = [next(r['coverage_pct'] for r in _rows_fair_T if r['method'] == m) for m in _order_cov]
    _y_cov = np.arange(len(_order_cov))
    _axes[2].barh(_y_cov - 0.2, _cov_all, height=0.4, color='#7f8c8d', alpha=0.5,
                   edgecolor='k', lw=0.5, label='all scope')
    _axes[2].barh(_y_cov + 0.2, _cov_fair, height=0.4, color='#2c3e50',
                   edgecolor='k', lw=0.5, label='fair scope')
    _axes[2].set_yticks(_y_cov); _axes[2].set_yticklabels(['' for _ in _order_cov])
    _axes[2].set_xlim(0, 105); _axes[2].set_xlabel('Coverage (%)')
    _axes[2].set_title('Coverage'); _axes[2].grid(True, alpha=0.3, axis='x')
    _axes[2].legend(loc='lower right', fontsize=7)
    _p5_plt.suptitle('Part 5: opx-only method comparison, dual-scope (alpha=all, opaque=fair)',
                      fontsize=12)
    _p5_plt.tight_layout()
    for _ext in ('png', 'pdf'):
        _p5_plt.savefig(FIGURES / f'fig_nb04_opx_only_comparison.{_ext}',
                         dpi=300, bbox_inches='tight')
    _p5_plt.close(_fig)


<!-- v7-fix-section-header -->
## Encyclopedia: per-method diagnostic pred-vs-obs

**Purpose.** Visual QC: one 1:1 scatter panel per method per target on the Part 2
ArcPL paired scope. Bad calibration, heteroscedasticity, or systematic bias
show up here before they get masked by summary statistics.

**Data inputs.** Uses `preds` dict from Part 2 (cell 24) on the paired scope.

**Methods evaluated.** All 13 methods from Part 2.

**Analysis performed.** For each method, plot predicted vs observed with 1:1
line. Color-code by family. Annotate with n, RMSE, R^2.

**How to interpret.** A clean 1:1 trend with tight scatter = well-calibrated.
Horizontal or L-shaped cloud = model reverting to mean. Vertical stripe at
one obs value = insufficient compositional spread.

**Outputs.**
- `figures/fig_nb04_diagnostic_encyclopedia_T.{png,pdf}`
- `figures/fig_nb04_diagnostic_encyclopedia_P.{png,pdf}`

**Downstream use.** Terminal outputs - manuscript supplementary material.


In [ ]:
# Diagnostic encyclopedia: 4x4 pred-vs-obs grids for T and P.
# Every benchmark method gets its own 1:1 scatter panel on the ArcPL
# three-phase paired scope (cpx + opx + liq). Family-colored, Putirka =
# square markers. 13 method panels + legend + summary + scope info.
# Self-contained: loads LEPR data and arcpl prediction manifest locally.
import joblib
import Thermobar as tb
from matplotlib.lines import Line2D

from src.plot_style import apply_style, OKABE_ITO
from src.data import canonical_model_path, canonical_model_spec
from src.features import build_feature_matrix, lepr_to_training_features
from src.external_models import (
    predict_agreda_from_df, predict_jorgenson, predict_wang,
    predict_putirka_cpx_liq, AGREDA_CPX_COLS,
)
from config import FAMILY_COLORS, LEPR_XLSX

apply_style()

# ---- Load LEPR cpx / liq / opx sheets (self-contained) ----
_xls = pd.ExcelFile(LEPR_XLSX)
_cpx_enc = pd.read_excel(_xls, sheet_name='Cpx')
_liq_enc = pd.read_excel(_xls, sheet_name='Liq')
_opx_enc = pd.read_excel(_xls, sheet_name='Opx')
def _numerify_enc(_df, _suffix):
    for _c in _df.columns:
        if _c.endswith(_suffix):
            _df[_c] = pd.to_numeric(_df[_c], errors='coerce').fillna(0.0)
    return _df
_cpx_enc = _numerify_enc(_cpx_enc.drop_duplicates('Experiment'), '_Cpx')
_liq_enc = _numerify_enc(_liq_enc.drop_duplicates('Experiment'), '_Liq')
_opx_enc = _numerify_enc(_opx_enc.drop_duplicates('Experiment'), '_Opx')
_arcpl_ids = pd.read_csv(RESULTS / 'nb04b_arcpl_predictions.csv')
keep_exp_enc = set(_arcpl_ids['Experiment'].astype(str))

# ---- Build ArcPL three-phase paired scope (cpx + opx + liq) ----
_scope = _cpx_enc.merge(_liq_enc, on='Experiment', how='inner',
                        suffixes=('', '_liq_dup'))
_scope = _scope.merge(_opx_enc, on='Experiment', how='inner',
                      suffixes=('', '_opx_dup'))
for _tgt in ['T_K', 'P_kbar']:
    for _src in ['', '_liq_dup', '_opx_dup']:
        _col = f'{_tgt}{_src}'
        if _col in _scope.columns:
            if _tgt not in _scope.columns:
                _scope[_tgt] = _scope[_col]
            else:
                _scope[_tgt] = _scope[_tgt].combine_first(_scope[_col])
_scope['T_C'] = pd.to_numeric(_scope['T_K'], errors='coerce') - 273.15
_scope['P_kbar'] = pd.to_numeric(_scope['P_kbar'], errors='coerce')
if 'Fe3Fet_Liq' not in _scope.columns:
    _scope['Fe3Fet_Liq'] = 0.0
_scope = _scope[_scope['Experiment'].astype(str).isin(keep_exp_enc)]
_has_cpx = _scope[AGREDA_CPX_COLS].apply(pd.to_numeric, errors='coerce').fillna(0.0).sum(axis=1) > 80
_scope = _scope[_has_cpx & np.isfinite(_scope['T_C']) & np.isfinite(_scope['P_kbar'])].reset_index(drop=True)
n_paired = len(_scope)
y_T_enc = _scope['T_C'].values
y_P_enc = _scope['P_kbar'].values

# ---- Training-schema copy for Ours inference ----
# LEPR uses SiO2_Opx / SiO2_Liq; the training schema uses SiO2 / liq_SiO2.
# Apply shim on a .copy() so _scope keeps its _Cpx/_Liq/_Opx columns for
# Thermobar + external ML methods.
_scope_train = lepr_to_training_features(_scope.copy())
for _fs in ['raw', 'pwlr', 'alr']:
    _Xt, _ = build_feature_matrix(_scope_train, _fs, use_liq=True)
    _Xo, _ = build_feature_matrix(_scope_train, _fs, use_liq=False)
    print(f'  feature check {_fs}: use_liq=True {_Xt.shape} '
          f'({int(np.isnan(_Xt).sum())} NaN) | use_liq=False {_Xo.shape} '
          f'({int(np.isnan(_Xo).sum())} NaN)')

FAMILY_MAP = {
    'Ours opx-liq forest':    'forest',
    'Ours opx-liq boosted':   'boosted',
    'Ours opx-only forest':   'forest',
    'Ours opx-only boosted':  'boosted',
    'Agreda-Lopez cpx-liq':   'external_cpx',
    'Agreda-Lopez cpx-only':  'external_cpx',
    'Jorgenson cpx-liq':      'external_cpx',
    'Jorgenson cpx-only':     'external_cpx',
    'Wang 2021 cpx-liq':      'external_cpx',
    'Wang 2021 cpx-only':     'external_cpx',
    'Putirka 2008 opx-liq':   'putirka',
    'Putirka 2008 opx-only':  'putirka',
    'Putirka 2008 cpx-liq':   'putirka',
    'Putirka 2008 cpx-only':  'putirka',
}

# ---- Compute every method on _scope ----
enc_preds = {}  # name -> (T_pred, P_pred)

# Ours opx-liq forest / boosted (training schema via shim)
for _fam in ['forest', 'boosted']:
    try:
        _sT = canonical_model_spec('T_C', 'opx_liq', _fam, RESULTS)
        _sP = canonical_model_spec('P_kbar', 'opx_liq', _fam, RESULTS)
        _mT = joblib.load(canonical_model_path('T_C', 'opx_liq', _fam, MODELS, RESULTS))
        _mP = joblib.load(canonical_model_path('P_kbar', 'opx_liq', _fam, MODELS, RESULTS))
        _Xt, _ = build_feature_matrix(_scope_train, _sT['feature_set'], use_liq=True)
        _Xp, _ = build_feature_matrix(_scope_train, _sP['feature_set'], use_liq=True)
        enc_preds[f'Ours opx-liq {_fam}'] = (_mT.predict(_Xt), _mP.predict(_Xp))
    except Exception as e:
        print(f'  Ours opx-liq {_fam} skipped: {e}')

# Ours opx-only forest / boosted (training schema via shim)
for _fam in ['forest', 'boosted']:
    try:
        _sT = canonical_model_spec('T_C', 'opx_only', _fam, RESULTS)
        _sP = canonical_model_spec('P_kbar', 'opx_only', _fam, RESULTS)
        _mT = joblib.load(canonical_model_path('T_C', 'opx_only', _fam, MODELS, RESULTS))
        _mP = joblib.load(canonical_model_path('P_kbar', 'opx_only', _fam, MODELS, RESULTS))
        _Xt, _ = build_feature_matrix(_scope_train, _sT['feature_set'], use_liq=False)
        _Xp, _ = build_feature_matrix(_scope_train, _sP['feature_set'], use_liq=False)
        enc_preds[f'Ours opx-only {_fam}'] = (_mT.predict(_Xt), _mP.predict(_Xp))
    except Exception as e:
        print(f'  Ours opx-only {_fam} skipped: {e}')

# Agreda-Lopez cpx-only / cpx-liq
for _kind, _label in [('cpx_only', 'cpx-only'), ('cpx_liq', 'cpx-liq')]:
    try:
        _aT = predict_agreda_from_df(_scope, MODELS / 'external', _kind, 'T')['median']
        _aP = predict_agreda_from_df(_scope, MODELS / 'external', _kind, 'P')['median']
        enc_preds[f'Agreda-Lopez {_label}'] = (_aT, _aP)
    except Exception as e:
        print(f'  Agreda {_kind} skipped: {e}')

# Jorgenson cpx-only / cpx-liq (Thermobar needs Sample_ID_Liq + NiO/CoO/CO2_Liq present)
_scope_jorg = _scope.copy()
for _c in ['Sample_ID_Liq', 'NiO_Liq', 'CoO_Liq', 'CO2_Liq']:
    if _c not in _scope_jorg.columns:
        _scope_jorg[_c] = 0.0 if _c != 'Sample_ID_Liq' else _scope_jorg['Experiment'].astype(str).values
for _phase, _label in [('cpx_only', 'cpx-only'), ('cpx_liq', 'cpx-liq')]:
    try:
        _jT = predict_jorgenson(_scope_jorg, 'T', phase=_phase, P_kbar=y_P_enc)
        _jP = predict_jorgenson(_scope_jorg, 'P', phase=_phase, T_K=y_T_enc + 273.15)
        enc_preds[f'Jorgenson {_label}'] = (_jT, _jP)
    except Exception as e:
        print(f'  Jorgenson {_phase} skipped: {e}')

# Wang 2021 cpx-liq
try:
    _wT = predict_wang(_scope, 'T', P_kbar=y_P_enc)
    _wP = predict_wang(_scope, 'P', T_K=y_T_enc + 273.15)
    enc_preds['Wang 2021 cpx-liq'] = (_wT, _wP)
except Exception as e:
    print(f'  Wang skipped: {e}')

# Putirka 2008 cpx-liq
try:
    _pT = predict_putirka_cpx_liq(_scope, 'T', P_kbar=y_P_enc)
    _pP = predict_putirka_cpx_liq(_scope, 'P', T_K=y_T_enc + 273.15)
    enc_preds['Putirka 2008 cpx-liq'] = (_pT, _pP)
except Exception as e:
    print(f'  Putirka cpx-liq skipped: {e}')

# Putirka 2008 cpx-only (eq32b P + eq32d T, iterative — eq32c is not a valid Thermobar id)
try:
    _cpx_df = _scope[[c for c in _scope.columns if c.endswith('_Cpx')]].apply(
        pd.to_numeric, errors='coerce').fillna(0.0)
    _out_cpx = tb.calculate_cpx_only_press_temp(
        cpx_comps=_cpx_df,
        equationP='P_Put2008_eq32b',
        equationT='T_Put2008_eq32d',
    )
    enc_preds['Putirka 2008 cpx-only'] = (
        np.asarray(_out_cpx['T_K_calc']) - 273.15,
        np.asarray(_out_cpx['P_kbar_calc']),
    )
except Exception as e:
    print(f'  Putirka cpx-only skipped: {e}')

# Wang 2021 cpx-only (direct Thermobar call; predict_wang is cpx-liq only)
try:
    _cpx_df_w = _scope[[c for c in _scope.columns if c.endswith('_Cpx')]].apply(
        pd.to_numeric, errors='coerce').fillna(0.0)
    _out_wco = tb.calculate_cpx_only_press_temp(
        cpx_comps=_cpx_df_w,
        equationP='P_Wang2021_eq1',
        equationT='T_Wang2021_eq2',
    )
    enc_preds['Wang 2021 cpx-only'] = (
        np.asarray(_out_wco['T_K_calc']) - 273.15,
        np.asarray(_out_wco['P_kbar_calc']),
    )
except Exception as e:
    print(f'  Wang 2021 cpx-only skipped: {e}')

# Putirka 2008 opx-liq (eq28a T, eq29a P, iterative)
try:
    _opx_cols = [c for c in _scope.columns if c.endswith('_Opx')]
    _liq_cols = [c for c in _scope.columns if c.endswith('_Liq')]
    _opx_df = _scope[_opx_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    _liq_df = _scope[_liq_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    _out_ol = tb.calculate_opx_liq_press_temp(
        opx_comps=_opx_df, liq_comps=_liq_df,
        equationP='P_Put2008_eq29a', equationT='T_Put2008_eq28a',
    )
    enc_preds['Putirka 2008 opx-liq'] = (
        np.asarray(_out_ol['T_K_calc']) - 273.15,
        np.asarray(_out_ol['P_kbar_calc']),
    )
except Exception as e:
    print(f'  Putirka opx-liq skipped: {e}')

# Putirka 2008 opx-only (eq28b_opx_sat T with P_obs, eq29c P with T_obs)
try:
    _opx_cols = [c for c in _scope.columns if c.endswith('_Opx')]
    _liq_cols = [c for c in _scope.columns if c.endswith('_Liq')]
    _opx_df = _scope[_opx_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    _liq_df = _scope[_liq_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    _out_T = tb.calculate_opx_liq_temp(
        equationT='T_Put2008_eq28b_opx_sat',
        opx_comps=_opx_df, liq_comps=_liq_df,
        P=y_P_enc,
    )
    _out_P = tb.calculate_opx_only_press(
        opx_comps=_opx_df, equationP='P_Put2008_eq29c',
        T=y_T_enc + 273.15,
    )
    # Thermobar returns DataFrame or Series depending on eq_tests
    def _as_T_C(obj):
        if isinstance(obj, pd.DataFrame):
            col = 'T_K_calc' if 'T_K_calc' in obj.columns else obj.columns[0]
            return np.asarray(obj[col]) - 273.15
        arr = np.asarray(obj)
        return arr - 273.15 if np.nanmedian(arr) > 400 else arr
    def _as_P_kbar(obj):
        if isinstance(obj, pd.DataFrame):
            col = 'P_kbar_calc' if 'P_kbar_calc' in obj.columns else obj.columns[0]
            return np.asarray(obj[col])
        return np.asarray(obj)
    enc_preds['Putirka 2008 opx-only'] = (_as_T_C(_out_T), _as_P_kbar(_out_P))
except Exception as e:
    print(f'  Putirka opx-only skipped: {e}')

print(f'Methods computed ({len(enc_preds)}):')
for _k in enc_preds:
    print(f'  - {_k}')

# ---- Render 4x4 grids ----
LAYOUT = [
    ('Ours opx-liq forest',  'Ours opx-liq boosted',  'Putirka 2008 opx-liq',  'Agreda-Lopez cpx-liq'),
    ('Ours opx-only forest', 'Ours opx-only boosted', 'Putirka 2008 opx-only', 'Jorgenson cpx-liq'),
    ('Putirka 2008 cpx-liq', 'Putirka 2008 cpx-only', 'Wang 2021 cpx-liq',     'Agreda-Lopez cpx-only'),
    ('Jorgenson cpx-only',   'Wang 2021 cpx-only',    '__LEGEND__',            '__SUMMARY__'),
]
T_LIMS = (700, 1700)
P_LIMS = (-2, 35)


def _enc_metrics(y, yhat):
    mask = np.isfinite(y) & np.isfinite(yhat)
    n = int(mask.sum())
    if n < 3:
        return {'n': n, 'rmse': np.nan, 'r2': np.nan, 'cov': 100 * n / len(y)}
    yt = y[mask]; yp = yhat[mask]
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))
    ss_r = float(np.sum((yt - yp) ** 2))
    ss_t = float(np.sum((yt - yt.mean()) ** 2))
    r2 = 1 - ss_r / ss_t if ss_t > 0 else np.nan
    return {'n': n, 'rmse': rmse, 'r2': r2, 'cov': 100 * n / len(y)}


def _is_put(name):
    return name.startswith('Putirka')


def _render_grid(obs, pred_idx, lims, unit, fmt_rmse, out_stem, suptitle):
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    off_notes = []
    low_cov_notes = []
    summary = []
    for r in range(4):
        for c in range(4):
            name = LAYOUT[r][c]
            ax = axes[r, c]
            if name.startswith('__'):
                continue
            if name not in enc_preds:
                ax.text(0.5, 0.5, f'{name}\n(unavailable)',
                        ha='center', va='center', transform=ax.transAxes,
                        fontsize=9, color='gray', style='italic')
                ax.set_xticks([]); ax.set_yticks([])
                continue
            yhat = np.asarray(enc_preds[name][pred_idx], dtype=float)
            m = _enc_metrics(obs, yhat)
            fam = FAMILY_MAP.get(name, 'putirka' if _is_put(name) else 'external_cpx')
            color = FAMILY_COLORS.get(fam, OKABE_ITO['yellow'])
            marker = 's' if _is_put(name) else 'o'
            mask = np.isfinite(obs) & np.isfinite(yhat)
            ax.scatter(obs[mask], yhat[mask], s=22, alpha=0.55,
                       c=color, edgecolor='k', linewidths=0.3, marker=marker)
            off = int(((yhat[mask] < lims[0]) | (yhat[mask] > lims[1])).sum())
            if off > 0:
                off_notes.append(f'{name}: {off}/{int(mask.sum())} off-axis')
            ax.plot(lims, lims, '--', color='gray', lw=0.9, alpha=0.8)
            ax.set_xlim(lims); ax.set_ylim(lims)
            ax.set_xlabel(f'Observed ({unit})', fontsize=9)
            ax.set_ylabel(f'Predicted ({unit})', fontsize=9)
            title = (f'{name}\n'
                     f'RMSE {fmt_rmse.format(m["rmse"])} {unit}  '
                     f'R$^2$={m["r2"]:+.2f}  n={m["n"]}')
            if m['cov'] < 95:
                title += f'\n{m["cov"]:.0f}% coverage'
                low_cov_notes.append(f'{name}: {m["cov"]:.0f}% coverage')
            ax.set_title(title, fontsize=8.5)
            ax.grid(True, alpha=0.3)
            ax.tick_params(labelsize=8)
            summary.append({**m, 'name': name})
    # Legend panel (3,2)
    ax = axes[3, 2]
    for side in ('top', 'right', 'bottom', 'left'):
        ax.spines[side].set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    handles = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=FAMILY_COLORS['forest'],
               markeredgecolor='k', markersize=11, label='Forest (RF/ERT) - Ours'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor=FAMILY_COLORS['boosted'],
               markeredgecolor='k', markersize=11, label='Boosted (GB/XGB) - Ours'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor=FAMILY_COLORS['external_cpx'],
               markeredgecolor='k', markersize=11, label='External ML (cpx)'),
        Line2D([0], [0], marker='s', color='w', markerfacecolor=FAMILY_COLORS['putirka'],
               markeredgecolor='k', markersize=11, label='Putirka 2008 (classical)'),
        Line2D([0], [0], linestyle='--', color='gray', label='1:1 line'),
    ]
    ax.legend(handles=handles, loc='center', fontsize=10, frameon=False)
    ax.set_title('Legend', fontsize=11, fontweight='bold')
    # Summary panel (3,3) - top 6 by RMSE + scope info merged in
    ax = axes[3, 3]
    for side in ('top', 'right', 'bottom', 'left'):
        ax.spines[side].set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    txt = (f'Scope\n{"-" * 28}\n'
           f'ArcPL three-phase paired\n(cpx + opx + liq)\n'
           f'n_paired = {n_paired}\n'
           f'T range: {T_LIMS[0]}-{T_LIMS[1]} C\n'
           f'P range: {P_LIMS[0]}-{P_LIMS[1]} kbar\n\n')
    if summary:
        sdf = pd.DataFrame(summary).sort_values('rmse', na_position='last')
        txt += 'Top 6 by RMSE\n' + '-' * 28 + '\n'
        for _, row in sdf.head(6).iterrows():
            name_short = row['name'].replace('Putirka 2008 ', 'Put. ').replace('Agreda-Lopez ', 'Agreda ').replace('Jorgenson ', 'Jorg. ').replace('Wang 2021 ', 'Wang ')
            txt += f"{name_short[:20]:<20}{fmt_rmse.format(row['rmse']):>8}\n"
    ax.text(0.02, 0.97, txt, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', family='monospace')
    ax.set_title('Scope + top methods', fontsize=11, fontweight='bold')

    fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.985])
    for ext in ('png', 'pdf'):
        fp = FIGURES / f'{out_stem}.{ext}'
        fig.savefig(fp, dpi=300 if ext == 'png' else None, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'\nSaved {out_stem}.')
    print(f'  Off-axis: {off_notes or ["none"]}')
    print(f'  <95% coverage: {low_cov_notes or ["none"]}')


_render_grid(y_T_enc, 0, T_LIMS, 'C', '{:.0f}',
             'fig_nb04_diagnostic_encyclopedia_T',
             f'Diagnostic encyclopedia: Temperature pred vs obs '
             f'(ArcPL three-phase paired, n={n_paired})')
_render_grid(y_P_enc, 1, P_LIMS, 'kbar', '{:.2f}',
             'fig_nb04_diagnostic_encyclopedia_P',
             f'Diagnostic encyclopedia: Pressure pred vs obs '
             f'(ArcPL three-phase paired, n={n_paired})')


<!-- v7-fix-section-header -->
## OPERATOR DECISION

**Purpose.** Headline framing for the benchmark — who wins on the internal
test set and under what caveats. This is the cell that determines the
one-line claim in the abstract.

**Data inputs.** Prints derivatives of `results/nb04_unified_benchmark.csv`
(from Part 1).

**Methods evaluated.** None new; summarizes Part 1 comparisons.

**Analysis performed.** Prints the headline RMSE + bootstrap CI for ML vs
Putirka (Option 1 iterative and Option 2 true P/T), plus the failure rate
gap. No decisions are auto-made; human reviewer chooses the framing.

**How to interpret.** This is a framing aid — the numbers it prints are
what the manuscript's one-sentence claim rests on.

**Outputs.** Printed summary only. No files written here.

**Downstream use.** Reviewer copies headline numbers into the manuscript
abstract and Section 4.


In [ ]:
# Part 1e: OPERATOR DECISION block - headline framing.
# This cell prints a decision prompt and does NOT choose for you. The manuscript
# headline for NB04 depends on which framing you prefer. Pick one of A/B/C and
# then tell Claude which to keep; the other cells can be trimmed accordingly.

ml_T_full = float(np.sqrt(np.mean((y_T_true - T_ml_pred) ** 2)))
ml_P_full = float(np.sqrt(np.mean((y_P_true - P_ml_pred) ** 2)))

row_T_inter = unified_df[(unified_df.target == 'T') &
                         (unified_df.scope == 'intersection')]
row_P_inter = unified_df[(unified_df.target == 'P') &
                         (unified_df.scope == 'intersection')]
pu_T_inter = row_T_inter[row_T_inter.method == 'Putirka 28a (true P)'].iloc[0]
pu_P_inter = row_P_inter[row_P_inter.method == 'Putirka 29a (true T)'].iloc[0]
ml_T_inter = row_T_inter[row_T_inter.method == 'ML-RF opx-liq'].iloc[0]
ml_P_inter = row_P_inter[row_P_inter.method == 'ML-RF opx-liq'].iloc[0]

w_T = wilcoxon_df[(wilcoxon_df.target == 'T') &
                  (wilcoxon_df.ml_vs == 'Putirka 28a (true P)')].iloc[0]
w_P = wilcoxon_df[(wilcoxon_df.target == 'P') &
                  (wilcoxon_df.ml_vs == 'Putirka 29a (true T)')].iloc[0]

fail_rate_T = 100 * (~np.isfinite(T_put_28a)).sum() / N_full
fail_rate_P = 100 * (~np.isfinite(P_put_29a)).sum() / N_full

print('=' * 72)
print('OPERATOR DECISION REQUIRED - NB04 headline framing')
print('=' * 72)
print()
print('Key numbers (opx-liq test set, N_full = {}):'.format(N_full))
print(f'  ML RMSE (full):         T {ml_T_full:.1f} C | P {ml_P_full:.2f} kbar')
print(f'  Intersection scope (n_T={int(ml_T_inter["n"])}, n_P={int(ml_P_inter["n"])}):')
print(f'    ML:           T {ml_T_inter["rmse"]:.1f} C [{ml_T_inter["rmse_ci_lo"]:.1f}, {ml_T_inter["rmse_ci_hi"]:.1f}]')
print(f'                  P {ml_P_inter["rmse"]:.2f} kbar [{ml_P_inter["rmse_ci_lo"]:.2f}, {ml_P_inter["rmse_ci_hi"]:.2f}]')
print(f'    Putirka28a/29a T {pu_T_inter["rmse"]:.1f} C [{pu_T_inter["rmse_ci_lo"]:.1f}, {pu_T_inter["rmse_ci_hi"]:.1f}]')
print(f'                   P {pu_P_inter["rmse"]:.2f} kbar [{pu_P_inter["rmse_ci_lo"]:.2f}, {pu_P_inter["rmse_ci_hi"]:.2f}]')
print(f'  Putirka failure rate:   T {fail_rate_T:.1f}% | P {fail_rate_P:.1f}%')
print(f'  Paired Wilcoxon p:      T {w_T["p_value"]:.4g} | P {w_P["p_value"]:.4g}')
print(f'  Paired Delta-RMSE 95% CI:')
print(f'    T [{w_T["delta_ci_lo"]:+.2f}, {w_T["delta_ci_hi"]:+.2f}] C   (> 0 => ML better)')
print(f'    P [{w_P["delta_ci_lo"]:+.3f}, {w_P["delta_ci_hi"]:+.3f}] kbar')
print()
print('Framing options:')
print('  A. "ML + broader applicability": lead with ML always-works vs')
print('     Putirka failure rate. Intersection RMSE is secondary (comparable).')
print('  B. "ML reduces error": lead with the intersection-scope RMSE + Wilcoxon')
print('     p-value, with failure rate as a supporting point.')
print('  C. "Hybrid": lead with the unified table (all three scopes), then show')
print('     the 4-panel figure, and let the reader read RMSE + failure rate.')
print()
print('Tell Claude which framing (A / B / C) to commit to in the manuscript body.')
print('No downstream cell auto-selects; the CSVs and figure are all written.')
print('=' * 72)
